# Siemens Advanta- Bussines Case Project 2025/2026

**Project developed by Group V**:
   - Alano Gonçalves (20250457)
   - Catarina Martins (20221914)
   - João Carichas (20250507)
   - Marta Ribeiro (20221886)
   - Nicole Nogueira(20221961)

## 1. Import the needed libraries

In [ ]:
#!pip install statsforecast
#!pip install prophet
#!pip install lightgbm


: 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from math import ceil 
import zipfile
import xml.etree.ElementTree as ET
from statsforecast import StatsForecast
from statsforecast.models import SeasonalNaive, AutoETS
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from statsmodels.tsa.statespace.sarimax import SARIMAX
from prophet import Prophet
import lightgbm as lgb
import logging
import random
from sklearn.preprocessing import LabelEncoder


In [ ]:
#import statsforecast
#print(statsforecast.__version__)

#from statsforecast import models
#print([x for x in dir(models) if 'ETS' in x or 'ets' in x.lower()])

## 2. Data importation

In [ ]:
def read_sales_sheet(zip_path, sheet_xml):
    with zipfile.ZipFile(zip_path) as z:
        with z.open('xl/sharedStrings.xml') as f:
            ss_root = ET.parse(f).getroot()
        ns = {'ns': 'http://purl.oclc.org/ooxml/spreadsheetml/main'}
        strings = []
        for si in ss_root.findall('.//ns:si', ns):
            texts = [t.text for t in si.findall('.//ns:t', ns) if t.text]
            strings.append(''.join(texts))

        with z.open(sheet_xml) as f:
            sheet_root = ET.parse(f).getroot()
        rows = []
        for row in sheet_root.findall('.//ns:row', ns):
            row_data = []
            for cell in row.findall('ns:c', ns):
                v = cell.find('ns:v', ns)
                t = cell.get('t')
                if v is not None:
                    row_data.append(strings[int(v.text)] if t == 's' else v.text)
                else:
                    row_data.append(None)
            rows.append(row_data)

    max_len = max(len(r) for r in rows)
    rows = [r + [None] * (max_len - len(r)) for r in rows]
    return pd.DataFrame(rows[1:], columns=rows[0])

In [ ]:
SALES_PATH  = 'Case2_data_extract_share.xlsx'
MARKET_PATH = 'Case2_market_data_share.xlsx'
train_data = read_sales_sheet(SALES_PATH, 'xl/worksheets/sheet1.xml')
test_data = read_sales_sheet(SALES_PATH, 'xl/worksheets/sheet3.xml')

In [ ]:
df_market = pd.read_excel(MARKET_PATH, sheet_name='Sheet1')
df_period_map = pd.read_excel(MARKET_PATH, sheet_name='Sheet2')

## 3. Data exploration and understanding

### 3.1. Changing Names and Datatypes

In [ ]:
#overview the dataset
train_data.info()

- At first sight, the training dataset has no missing values and have 4237 rows. 
- The datatypes, are not correct so we will have to fix them. 

In [ ]:
#changing the name of the variables do it is easy to track them
train_data = train_data.rename(columns={
    'Anon Period': 'Period',
    'TGL Biz Desc': 'Biz_Desc',
    'TGL Business Unit': 'Business_Unit',
    'TGL Business Segment': 'Segment',
    'TGL Business Subsegment': 'Subsegment',
    'Orders cons. (anon)': 'Orders',
    'Revenue cons. (anon)': 'Revenue'
})

In [ ]:
#changing the datatypes so it is possible to visualize the data
train_data['Period'] = train_data['Period'].round().astype('Int32')
train_data['Orders'] = train_data['Orders'].round().astype('Int32')
train_data['Revenue'] = train_data['Revenue'].round().astype('Int32')

### 3.2. Data Overview

In [ ]:
#overview the dataset
train_data.info()

In [ ]:
df_market.info()

- Considering that the total number of rows is 180 (corresponding to the number of years), most of the columns have a considerable amount of missing values.

In [ ]:
#first 20 rows
train_data.head(20)

In [ ]:
#first 20 rows
df_market.head(20)

In [ ]:
#descriptive statistics for numerical data
train_data.describe()

- The dataset has 42 months.
- There are rows in *Orders* and *Revenue* that are negative- Is it possible?

In [ ]:
#descriptive statistics for numerical data
df_market.describe().round(2)

- Unlike *train_data*, the period goes form -131 to 48. 

In [ ]:
train_data.nunique()

In [ ]:
df_market.nunique()

### 3.3. Checking Duplicates

In [ ]:
#checking number of duplicates
train_data.duplicated().sum()

In [ ]:
#checking number of duplicates
df_market.duplicated().sum()

- None of the datasets present duplicates.

### 3.4. Checking Missing Values

In [ ]:
#checking number of missing values
train_data.isna().sum()

- The train dataset does not present any missing values which is a good sign.

In [ ]:
#checking number of missing values
df_market.isna().sum()

In [ ]:
pd.set_option('display.max_rows', None)
df_market.isna().sum()/len(df_market) * 100

- The variables whose missing values represent more than 50% of the data correspond to columns that have informatio about the GDP
Some columns present a high amount of missing values- almost 92% of the total data. In those cases, the information given is not sufficiently significant to consider these variables.
- 

- Data that have more than 50% of missing observations correspond to the GDP columns of each country (for example, China_GDP, China_GDP_from_Construction, China_GDP_from_Manufacturing, etc). In these cases, it is normal for the GDP to have a high quantity of missing values considering that the GDP of each country can only be calculated at the end of each year and since we have our data per month, this is the usual behavior. 
- The rest of the columns have less than 50% of missing values (usually the values do not exceed 5%, except for steel production).For the columns corresponding to steel production, there are some countries that for the possitive months (the ones we are going to consider later on merge) have 100% of missing values. 

In [ ]:
#creating a variable that has df_market data from the 1 to 42 months 
df_market_months = df_market[df_market['Period'].between(1, 42)]

In [ ]:
pd.set_option('display.max_rows', None)
df_market_months.isna().sum()/len(df_market_months) * 100

- If we consider only the observations whose period we are going to consider when merging with *train_data*, some of these observations have 100% missing values. In those cases, it is useless to consider them. 
- All variables that have missing values, also have a high percentage (>60%), except for *United_States_Interest_Rate* that has only 2.3% and *China_Industrial_Production* that has 2.3%. Therefore, these are the only variables worth of handling missing values.

### 3.5. Checking Outliers

In [ ]:
#setting metric fearures for train_data
metric_features_train= ['Revenue', 'Orders']

In [ ]:
#setting metric fearures for df_market
metric_features_market= ['China_Core_Inflation_Rate', 'China_Exports', 'China_GDP', 
                         'China_Industrial_Production', 'China_Industrial_Production_Mom', 
                         'China_Inflation_Rate', 'China_Interest_Rate', 'China_Steel_Production', 
                         'France_Core_Inflation_Rate', 'France_Exports', 'France_Industrial_Production',
                         'France_Industrial_Production_Mom', 'France_Inflation_Rate', 'France_Interest_Rate', 
                         'Germany_Core_Inflation_Rate', 'Germany_Exports', 'Germany_Industrial_Production', 
                         'Germany_Industrial_Production_Mom', 'Germany_Inflation_Rate', 'Germany_Interest_Rate', 
                         'Germany_Steel_Production', 'Italy_Core_Inflation_Rate', 'Italy_Exports', 'Italy_Industrial_Production',
                         'Italy_Industrial_Production_Mom', 'Italy_Inflation_Rate', 'Italy_Interest_Rate', 'Japan_Core_Inflation_Rate',
                         'Japan_Exports', 'Japan_Industrial_Production', 'Japan_Industrial_Production_Mom', 'Japan_Inflation_Rate', 
                         'Japan_Interest_Rate', 'Japan_Steel_Production', 'Switzerland_Core_Inflation_Rate', 'Switzerland_Exports', 
                         'Switzerland_Inflation_Rate', 'Switzerland_Interest_Rate', 'United_Kingdom_Core_Inflation_Rate', 'United_Kingdom_Exports', 
                         'United_Kingdom_Industrial_Production', 'United_Kingdom_Industrial_Production_Mom', 'United_Kingdom_Inflation_Rate', 
                         'United_Kingdom_Interest_Rate', 'United_States_Core_Inflation_Rate', 'United_States_Exports', 
                         'United_States_Industrial_Production', 'United_States_Industrial_Production_Mom', 'United_States_Inflation_Rate', 
                         'United_States_Steel_Production'
]

In [ ]:
def plot_multiple_boxplots(data, feats, n_cols=2, title="Numeric Variables - Box Plots"):

    #defining grid size
    n_rows = ceil(len(feats) / n_cols)

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(6 * n_cols, 4.5 * n_rows)
    )

    axes = axes.flatten()
    #looping through features
    for i, feat in enumerate(feats):

        #keeping ALL values except missing ones
        values = data[feat].dropna()

        #plotting only if there is data
        if len(values) > 0:
            sns.boxplot(x=values, ax=axes[i], color="#798a40")
            axes[i].set_title(feat, fontsize=10)
        else:
            axes[i].set_title(f"{feat} (no data)", fontsize=10)

    # removing empty subplots
    for j in range(len(feats), len(axes)):
        fig.delaxes(axes[j])

    # final layout adjustments
    plt.suptitle(title, fontsize=18)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()

In [ ]:
#applying the function to our numerical features
plot_multiple_boxplots(train_data, metric_features_train)

In [ ]:
def plot_multiple_boxplots(data, feats, n_cols=3, title="Numeric Variables - Box Plots"):

    #defining grid size
    n_rows = ceil(len(feats) / n_cols)

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(6 * n_cols, 4.5 * n_rows)
    )

    axes = axes.flatten()
    #looping through features
    for i, feat in enumerate(feats):

        #keeping ALL values except missing ones
        values = data[feat].dropna()

        #plotting only if there is data
        if len(values) > 0:
            sns.boxplot(x=values, ax=axes[i], color="#798a40")
            axes[i].set_title(feat, fontsize=10)
        else:
            axes[i].set_title(f"{feat} (no data)", fontsize=10)

    # removing empty subplots
    for j in range(len(feats), len(axes)):
        fig.delaxes(axes[j])

    # final layout adjustments
    plt.suptitle(title, fontsize=18)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()

In [ ]:
#applying the function to our numerical features
plot_multiple_boxplots(df_market, metric_features_market)

- Even tough there are some outliers, these values correspond to real macroeconomic indicators (like inflation, post-COVID period, etc.). If we applied some capping or transformation to try handle them, we would be distorting the actual economic reality that your model needs to learn from. Therefore, we have decided to keep the data as it is for now. 

### 3.6. Checking Distribution

In [ ]:
df= pd.DataFrame(train_data[metric_features_train])

In [ ]:
#setting visual style
sns.set_style("white")  # clean background for reports
#sampling the dataset
N = 200_000
sample = train_data.sample(n=min(N, len(train_data)), random_state=42)

#selecting numeric variables
cols = metric_features_train

#defining plotting structure
plots_per_fig = 6   
n_figs = ceil(len(cols) / plots_per_fig)

#looping through figures
for f in range(n_figs):
    
    #selecting subset of columns for current figure
    start = f * plots_per_fig
    end = start + plots_per_fig
    subset_cols = cols[start:end]
    
    #creating subplot grid
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    axes = axes.ravel()
    
    #ploting histograms
    for ax, col in zip(axes, subset_cols):
        
        #removing missing values only 
        data = sample[col].dropna()
        
        sns.histplot(
            data,
            bins=60,
            kde=False,
            ax=ax,
            color="#798a40"   
        )
        
        ax.set_title(col, fontsize=10)

    #removing empty subplots
    for j in range(len(subset_cols), len(axes)):
        fig.delaxes(axes[j])
    
        #adjusting layout and display
    plt.tight_layout()
    plt.show()

In [ ]:
#setting visual style
sns.set_style("white")  # clean background for reports
#sampling the dataset
N = 200_000
sample = df_market.sample(n=min(N, len(df_market)), random_state=42)

#selecting numeric variables
cols = metric_features_market

#defining plotting structure
plots_per_fig = 6   
n_figs = ceil(len(cols) / plots_per_fig)

#looping through figures
for f in range(n_figs):
    
    #selecting subset of columns for current figure
    start = f * plots_per_fig
    end = start + plots_per_fig
    subset_cols = cols[start:end]
    
    #creating subplot grid
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    axes = axes.ravel()
    
    #ploting histograms
    for ax, col in zip(axes, subset_cols):
        
        #removing missing values only 
        data = sample[col].dropna()
        
        sns.histplot(
            data,
            bins=60,
            kde=False,
            ax=ax,
            color="#798a40"   
        )
        
        ax.set_title(col, fontsize=10)

    #removing empty subplots
    for j in range(len(subset_cols), len(axes)):
        fig.delaxes(axes[j])
    
        #adjusting layout and display
    plt.tight_layout()
    plt.show()

### 3.7. Checking Correlation between variables

In [ ]:
#checking correlation between variables 
#we are going to use spearman correlation since our variables do not follow a normal distribution
cor_spearman = df_market[metric_features_market].corr(method ='spearman')
cor_spearman

In [ ]:
#creating correlation matrix to facilitate interpretation
def cor_heatmap(cor):
    
    #setting the figure size
    plt.figure(figsize=(12, 10))

    #creating a mask for the upper triangle of the matrix (to avoid plotting duplicate correlation values)
    mask = np.triu(np.ones_like(cor, dtype=bool))

    #plotting the correlation heatmap
    sns.heatmap(
        data=cor,                 #correlation matrix input
        mask=mask,                #applying upper-triangle mask
        annot=True,               #displaying correlation coefficients
        cmap=sns.light_palette("#798a40", as_cmap=True), #color map for visual contrast
        fmt='.2f',                #formatting values to two decimals
        square=True,              #ensuring square-shaped cells
        linewidths=0.5,           #adding grid lines between cells
        cbar_kws={"shrink": 0.8}  #adjusting color bar size
    )

    #adding a title and display the plot
    plt.title("Spearman Correlation Matrix", fontsize=14)
    plt.show()

In [ ]:
#applying the function to our numerical features
cor_heatmap(cor_spearman)

## 4. Data Preparation

### 4.1. Handling Missing Values

- For columns with >50% missing values (corresponding to GDP columns), we have decided to fill the missing observations with the value of the GDP of that year. 
- For columns with <50% missing values, interpolation is going to be applied in order to fill them. 
- Columns whose missing values reach 100% in positive months (corresponding to steel production of some countries), we have decided to drop those columns, since it does not contain relevant information that could be used in the model later on. 

In [ ]:
# columns to drop - steel production countries with 100% missing in periods 1-48
cols_to_drop = [
    'France_Steel_Production',
    'Italy_Steel_Production',
    'United_Kingdom_Steel_Production'
]
df_market = df_market.drop(columns=cols_to_drop)

# identify GDP columns (more than 50% missing) vs other columns
missing_pct = df_market.isna().sum() / len(df_market) * 100
gdp_cols    = missing_pct[missing_pct > 50].index.tolist()
other_cols  = missing_pct[(missing_pct > 0) & (missing_pct <= 50)].index.tolist()

print("GDP columns to forward-fill:", gdp_cols)
print("Other columns to interpolate:", other_cols)

# forward-fill GDP columns, then backward-fill any remaining NaNs at the start
df_market = df_market.sort_values('Period').reset_index(drop=True)
df_market[gdp_cols] = df_market[gdp_cols].ffill().bfill()

# linear interpolation for columns with less than 50% missing + ffill/bfill for edges
df_market[other_cols] = df_market[other_cols].interpolate(method='linear').ffill().bfill()

# verify no missing values remain
remaining = df_market.isna().sum()
remaining = remaining[remaining > 0]
print("\nRemaining missing values:")
print(remaining if len(remaining) > 0 else "None - all clean!")

In [ ]:
#checking number of missing values
df_market.isna().sum()

### 4.2. Merging Datasets

In [ ]:
#merging the datasets on the left
df_merged = train_data.merge(df_market, on='Period', how='left')

print(df_merged.shape)

In [ ]:
df_merged.head(25)

In [ ]:
df_merged.info()

In [ ]:
#checking missing values after merging 
pd.set_option('display.max_rows', None)
df_merged.isna().sum()/len(df_merged) * 100

In [ ]:
# add real dates from period map
df_period_map = df_period_map.dropna(subset=['Period'])
df_period_map['Period'] = df_period_map['Period'].astype(int)
df_merged = df_merged.merge(df_period_map[['Period', 'DATE']], on='Period', how='left')
df_merged = df_merged.rename(columns={'DATE': 'Date'})

print(df_merged[['Period', 'Date']].drop_duplicates().head(10))

In [ ]:
df_merged.head()

In [ ]:
#exporting dataset
dataset = pd.DataFrame(df_merged)
dataset.to_csv("df_merged.csv", index=False)

### 4.3 Creating our Reconciliation Datasets

Here, we will create a separate dataframe for every hierarchy level (BU, Segment and Subsegment):
- df_subseg for our Subsegment level;
- df_seg for our Segment level;
- df_bu for our Business Unit level.

First, we will specify our target, time column and hierarchy levels, then we will check to see possible incoherences, such as the same subsegment having diferent segments related to it (as well as that relation between segments and BUs).

In [ ]:
TARGET = 'Revenue'
TIME_COL = 'Period'
LEVELS = ['Business_Unit', 'Segment', 'Subsegment']

In [ ]:
df = df_merged.copy()

subseg_to_segment = (
    df[['Subsegment', 'Segment']]
    .drop_duplicates()
    .groupby('Subsegment')
    .size()
)

print((subseg_to_segment > 1).sum())

segment_to_bu = (
    df[['Segment', 'Business_Unit']]
    .drop_duplicates()
    .groupby('Segment')
    .size()
)

print((segment_to_bu > 1).sum())

According with the output above, we can see that we don't have a relationship problem between our hirarchy levels.

Now, we will exclude every subsegment that has a revenue value constantly equal to 0.

In [ ]:
df_base = df.copy()

all_periods = sorted(df_base[TIME_COL].unique())

In [ ]:
zero_only_subsegments = (
    df_base.groupby('Subsegment')[TARGET]
    .sum()
    .reset_index()
)

zero_only_subsegments = zero_only_subsegments.loc[
    zero_only_subsegments[TARGET] == 0, 'Subsegment'
].tolist()

print("Amount of Subsegments with a constant revenue of 0:", len(zero_only_subsegments))
print(zero_only_subsegments[:10])

Now, we will fill missing values in the timelines of every series (specifically in the 'Revenue' and 'Orders' columns), creating our dataset for subsegment level.

In [ ]:
macro_cols = [col for col in df_base.columns if col not in [TIME_COL, 'Business_Unit', 'Segment', 'Subsegment', 'Orders', 'Revenue']]

aligned_list = []

for subseg, grp in df_base.groupby('Subsegment'):
    
    if subseg in zero_only_subsegments:
        continue
    
    grp = grp.sort_values(TIME_COL)

    bu = grp['Business_Unit'].iloc[0]
    seg = grp['Segment'].iloc[0]

    grp = grp.set_index(TIME_COL).reindex(all_periods)

    grp['Business_Unit'] = bu
    grp['Segment'] = seg
    grp['Subsegment'] = subseg
    grp.index.name = TIME_COL

    grp[TARGET] = grp[TARGET].fillna(0)
    grp['Orders'] = grp['Orders'].fillna(0)

    for col in macro_cols:
        grp[col] = grp[col].ffill().bfill()

    aligned_list.append(grp.reset_index())

df_subseg = pd.concat(aligned_list, ignore_index=True)

In [ ]:
#exemple of a series
df_subseg.head(42)

Now, we will create our dataset for segment and BU levels, by aggregating.

In [ ]:
agg_dict_segment = {TARGET: 'sum',
                    'Orders': 'sum'}

for col in macro_cols:
    agg_dict_segment[col] = 'first'

df_seg = (
    df_subseg
    .groupby(['Business_Unit', 'Segment', TIME_COL], as_index=False)
    .agg(agg_dict_segment)
)

df_seg.head(42)

In [ ]:
agg_dict_bu = {TARGET: 'sum',
               'Orders': 'sum'}

for col in macro_cols:
    agg_dict_bu[col] = 'first'

df_bu = (
    df_subseg
    .groupby(['Business_Unit', TIME_COL], as_index=False)
    .agg(agg_dict_bu)
)

df_bu.head(42)

In [ ]:
agg_dict_total = {TARGET: 'sum',
                  'Orders': 'sum'}

for col in macro_cols:
    agg_dict_total[col] = 'first'

df_total = (
    df_subseg
    .groupby([TIME_COL], as_index=False)
    .agg(agg_dict_total)
)
df_total['node'] = 'TOTAL'

df_total.head(42)

### 4.4 Hierarchy Matrix

Let's create a diferenciator between the hierarchy levels.

In [ ]:
df_subseg['node'] = 'SUBSEG__' + df_subseg['Subsegment'].astype(str)
df_seg['node'] = ('SEG__' + df_seg['Business_Unit'].astype(str) + '__' + df_seg['Segment'].astype(str))
df_bu['node'] = 'BU__' + df_bu['Business_Unit'].astype(str)

In [ ]:
hierarchy_df = (
    df_subseg[['Business_Unit', 'Segment', 'Subsegment']]
    .drop_duplicates()
    .copy()
)

hierarchy_df['subseg_node'] = 'SUBSEG__' + hierarchy_df['Subsegment'].astype(str)
hierarchy_df['segment_node'] = (
    'SEG__' + hierarchy_df['Business_Unit'].astype(str) + '__' + hierarchy_df['Segment'].astype(str)
)
hierarchy_df['bu_node'] = 'BU__' + hierarchy_df['Business_Unit'].astype(str)
hierarchy_df['total_node'] = 'TOTAL'

In [ ]:
hierarchy_df.head()

In [ ]:
base_long = df_subseg[[TIME_COL, 'node', TARGET]].copy()
base_long['level'] = 'Subsegment'

segment_long = df_seg[[TIME_COL, 'node', TARGET]].copy()
segment_long['level'] = 'Segment'

bu_long = df_bu[[TIME_COL, 'node', TARGET]].copy()
bu_long['level'] = 'Business_Unit'

total_long = df_total[[TIME_COL, 'node', TARGET]].copy()
total_long['level'] = 'Total'

all_series_long = pd.concat(
    [base_long, segment_long, bu_long, total_long],
    ignore_index=True
    ).sort_values(['level', 'node', TIME_COL])

## 5. Starting Insights

### 5.1 Baselines Models (AutoETS and SeasonalNaives)

Before we start implementing our baseline models we will check to see how many series we have across the 3 hierarchy levels.

In [ ]:
series_dict = {
    node: df.sort_values('Period')['Revenue'].values
    for node, df in all_series_long.groupby('node')
}

len(series_dict)

163 series

In [ ]:
ts_df = all_series_long[['node', 'Period', 'Revenue']].copy()
ts_df = ts_df.rename(columns={
    'node': 'unique_id',
    'Period': 'ds',
    'Revenue': 'y'
})
ts_df = ts_df.sort_values(['unique_id', 'ds']).reset_index(drop=True)

train-val split temporal, deixando as 6 ultimas series como validation set.

In [ ]:
HORIZON = 6

Y_train = ts_df.groupby('unique_id').apply(lambda x: x.iloc[:-HORIZON]).reset_index(drop=True)
Y_val   = ts_df.groupby('unique_id').apply(lambda x: x.iloc[-HORIZON:]).reset_index(drop=True)

treinar ambos os modelos iniciais (AutoETS e SeasonalNaive).

In [ ]:
print("\n" + "="*60)
print("Base Models Training")
print("="*60)

models = [
    SeasonalNaive(season_length=12),
    AutoETS(season_length=12),
]

sf = StatsForecast(models=models, freq=1, n_jobs=-1)

print("Training... (1-2 min)")
sf.fit(Y_train)
print("Train OK!")

Y_hat_df = sf.predict(h=HORIZON).reset_index(drop = False)
print(f"Predictions: {Y_hat_df.shape}")

Evaluation dataset:

In [ ]:
eval_df = Y_val.merge(Y_hat_df, on=['unique_id', 'ds'], how='inner')
print(eval_df.shape)

if 'index' in eval_df.columns:
    eval_df = eval_df.drop(columns='index')

eval_df.head()

Checking our starting RMSE for each model:

In [ ]:
rmse_snaive = np.sqrt(mean_squared_error(eval_df['y'], eval_df['SeasonalNaive']))
print("RMSE global - SeasonalNaive:", rmse_snaive)

rmse_autoets = np.sqrt(mean_squared_error(eval_df['y'], eval_df['AutoETS']))
print("RMSE global - AutoETS:", rmse_autoets)

In [ ]:
rmse_by_series = []

for uid, grp in eval_df.groupby('unique_id'):
    rmse_s = np.sqrt(mean_squared_error(grp['y'], grp['SeasonalNaive']))
    rmse_a = np.sqrt(mean_squared_error(grp['y'], grp['AutoETS']))
    
    rmse_by_series.append({
        'unique_id': uid,
        'rmse_SeasonalNaive': rmse_s,
        'rmse_AutoETS': rmse_a
    })

rmse_by_series = pd.DataFrame(rmse_by_series)
rmse_by_series.head()

In [ ]:
wins_autoets = (rmse_by_series['rmse_AutoETS'] < rmse_by_series['rmse_SeasonalNaive']).sum()
wins_snaive  = (rmse_by_series['rmse_SeasonalNaive'] < rmse_by_series['rmse_AutoETS']).sum()
ties = ((rmse_by_series['rmse_AutoETS'] == rmse_by_series['rmse_SeasonalNaive']).sum())

print("Series with lower RMSE on AutoETS:", wins_autoets)
print("Series with lower RMSE on SeasonalNaives:", wins_snaive)
print("Series with equal RMSE on AutoETS and SeasonalNaives:", ties)

On the outputs above, we can see that, although AutoETS had a RMSE value of around 27M lower than SeasonalNaives, there were 56 series where SeasonalNaives performed better.

We can also see that 30 series had similar RMSE in both models.

### 5.2 Bottom-Up Reconciliation (AutoETS)

**(BOTTOM-UP ESTÁ A SER USADO COMO VALIDAÇÃO NESTA FASE, NAO SERÁ O FINAL)**

First, let's create a dataframe for our base forecast and separate the diferent hierarchy levels.

In [ ]:
#creating our base forecast dataset
base_fcst_df = eval_df[['unique_id', 'ds', 'y', 'AutoETS']].copy()
base_fcst_df = base_fcst_df.rename(columns={'AutoETS': 'base_forecast'})

#separating by the diferent hierarchy levels
subseg_nodes = sorted([uid for uid in base_fcst_df['unique_id'].unique() if uid.startswith('SUBSEG__')])
segment_nodes = sorted([uid for uid in base_fcst_df['unique_id'].unique() if uid.startswith('SEG__')])
bu_nodes = sorted([uid for uid in base_fcst_df['unique_id'].unique() if uid.startswith('BU__')])

mapping between levels

In [ ]:
#from subsegment to segment
subseg_to_segment = dict(zip(
    hierarchy_df['subseg_node'],
    hierarchy_df['segment_node']
))

#from segment to business unit
segment_to_bu = dict(zip(
    hierarchy_df['segment_node'],
    hierarchy_df['bu_node']
))

**passo 1:** filtrar por subsegmentos

In [ ]:
subseg_df = base_fcst_df[base_fcst_df['unique_id'].isin(subseg_nodes)].copy()

**passo 2:** calcular segmentos

In [ ]:
segment_recon = (
    subseg_df
    .assign(segment=lambda x: x['unique_id'].map(subseg_to_segment))
    .groupby(['segment', 'ds'])['base_forecast']
    .sum()
    .reset_index()
    .rename(columns={'segment': 'unique_id'})
)

**passo 3:** calcular bu

In [ ]:
bu_recon = (
    segment_recon
    .assign(bu=lambda x: x['unique_id'].map(segment_to_bu))
    .groupby(['bu', 'ds'])['base_forecast']
    .sum()
    .reset_index()
    .rename(columns={'bu': 'unique_id'})
)

**passo 4:** calcular total

In [ ]:
total_recon = (
    bu_recon
    .groupby(['ds'])['base_forecast']
    .sum()
    .reset_index()
)

total_recon['unique_id'] = 'TOTAL'

**passo 5:** juntar tudo

In [ ]:
reconciled_df = pd.concat([
    subseg_df[['unique_id', 'ds', 'base_forecast']],
    segment_recon,
    bu_recon,
    total_recon
], ignore_index=True)

Now, we will evaluate this reconciled_df:

In [ ]:
eval_recon = eval_df[['unique_id', 'ds', 'y']].merge(
    reconciled_df,
    on=['unique_id', 'ds'],
    how='inner'
)

rmse_recon = np.sqrt(mean_squared_error(eval_recon['y'], eval_recon['base_forecast']))
print("RMSE reconciled:", rmse_recon)

In [ ]:
rmse_base = np.sqrt(mean_squared_error(eval_df['y'], eval_df['AutoETS']))

print("RMSE base:", rmse_base)
print("RMSE reconciled:", rmse_recon)

In [ ]:
t = reconciled_df['ds'].iloc[0]

sub_sum = reconciled_df[
    (reconciled_df['ds'] == t) & 
    (reconciled_df['unique_id'].isin(subseg_nodes))
]['base_forecast'].sum()

total_val = reconciled_df[
    (reconciled_df['ds'] == t) & 
    (reconciled_df['unique_id'] == 'TOTAL')
]['base_forecast'].iloc[0]

print(sub_sum, total_val)

### 5.3 MinTrace OLS Reconciliation

In [ ]:
all_nodes_list = ['TOTAL'] + sorted(bu_nodes) + sorted(segment_nodes) + sorted(subseg_nodes)
 
n_bottom = len(subseg_nodes)   # 134 subsegments
n_total = len(all_nodes_list)  # 163 series
 
print(f"Total de series: {n_total}")
print(f"Series bottom (subsegments): {n_bottom}")

In [ ]:
# first we will create the matrix S
S = np.zeros((n_total, n_bottom))
 
# mapping the subsegments as indexes in the matrix
subseg_to_idx = {node: i for i, node in enumerate(sorted(subseg_nodes))}

In [ ]:
# filling the values in the matrix S
for i, node in enumerate(all_nodes_list):
    if node == 'TOTAL':
        # Total = soma de todos os subsegments
        S[i, :] = 1
    elif node.startswith('BU__'):
        # BU = soma dos subsegments dessa BU
        bu_name = node.replace('BU__', '')
        for subseg in subseg_nodes:
            # Verificar se este subseg pertence a esta BU
            if subseg in subseg_to_segment:
                seg = subseg_to_segment[subseg]
                if seg in segment_to_bu and segment_to_bu[seg] == node:
                    S[i, subseg_to_idx[subseg]] = 1
    elif node.startswith('SEG__'):
        # Segment = soma dos subsegments desse segment
        for subseg in subseg_nodes:
            if subseg in subseg_to_segment and subseg_to_segment[subseg] == node:
                S[i, subseg_to_idx[subseg]] = 1
    elif node.startswith('SUBSEG__'):
        # Subsegment = ele próprio (matriz identidade)
        S[i, subseg_to_idx[node]] = 1
 
S_df = pd.DataFrame(S, index=all_nodes_list, columns=sorted(subseg_nodes))
 
print(f"\nMatriz S shape: {S_df.shape}")
print(f"Verificação - soma da linha TOTAL: {int(S_df.loc['TOTAL'].sum())}")

After creating the X Matrix, we need to create our Reconciliation function and prepare our base predictions

In [ ]:
# Reconciliation MinTrace OLS
 
def reconcile_mintrace_ols(Y_hat_matrix, S_matrix):
    """
    Args:
        Y_hat_matrix: previsões base (n_series x n_horizons) - numpy array
        S_matrix: matriz de agregação (n_series x n_bottom) - numpy array
    
    Returns:
        Previsões reconciliadas (n_series x n_horizons)
    """
    # ensuring the numpy arrays
    if isinstance(Y_hat_matrix, pd.DataFrame):
        Y_hat_matrix = Y_hat_matrix.values
    if isinstance(S_matrix, pd.DataFrame):
        S_matrix = S_matrix.values
    
    # calculating the projection matrix
    StS = S_matrix.T @ S_matrix
    StS_inv = np.linalg.inv(StS)
    P = S_matrix @ StS_inv @ S_matrix.T
    
    # applying reconciliation
    Y_reconciled = P @ Y_hat_matrix
    
    return Y_reconciled

In [ ]:
# we will use the base predictions from the eval_df to all series (subsegments, segments, BUs, total)
 
# creating pivot (rows: series, columns: periods)
Y_hat_pivot = eval_df.pivot(index='unique_id', columns='ds', values='AutoETS')
 
# reordering to match the order of the S matrix
Y_hat_pivot = Y_hat_pivot.reindex(all_nodes_list)
 
print(f"\nPrevisões base shape: {Y_hat_pivot.shape}")
 
# checking for NaN values before reconciliation
nan_count = Y_hat_pivot.isna().sum().sum()
if nan_count > 0:
    print(f"{nan_count} valores NaN in the predictions")
else:
    print("No NaNs in the predictions")
 

Now we will apply the MinTrace OLS Reconciliation

In [ ]:
Y_reconciled_matrix = reconcile_mintrace_ols(Y_hat_pivot.values, S_df.values)
 
# converting to DataFrame
Y_reconciled_mintrace = pd.DataFrame(
    Y_reconciled_matrix,
    index=all_nodes_list,
    columns=Y_hat_pivot.columns
)

In [ ]:
Y_actual_pivot = eval_df.pivot(index='unique_id', columns='ds', values='y')
Y_actual_pivot = Y_actual_pivot.reindex(all_nodes_list)
 
# RMSE function
def calc_rmse(actual, predicted):
    # Converter para float (esta linha resolve o erro!)
    actual_flat = actual.values.flatten().astype(float)
    pred_flat = predicted.values.flatten().astype(float)
    
    mask = ~(np.isnan(actual_flat) | np.isnan(pred_flat))
    
    if mask.sum() == 0:
        return np.nan
    
    return np.sqrt(mean_squared_error(actual_flat[mask], pred_flat[mask]))
 
# RMSE Global
rmse_base = calc_rmse(Y_actual_pivot, Y_hat_pivot)
rmse_mintrace = calc_rmse(Y_actual_pivot, Y_reconciled_mintrace)

In [ ]:
# using the bottom-up forecasts in the same format for comparison
Y_bottomup_pivot = reconciled_df.pivot(index='unique_id', columns='ds', values='base_forecast')
Y_bottomup_pivot = Y_bottomup_pivot.reindex(all_nodes_list)

# calculate the RMSE of the 3 methods
rmse_base = calc_rmse(Y_actual_pivot, Y_hat_pivot)
rmse_bottomup = calc_rmse(Y_actual_pivot, Y_bottomup_pivot)
rmse_mintrace = calc_rmse(Y_actual_pivot, Y_reconciled_mintrace)

print(f"RMSE Base:      {rmse_base:,.0f}")
print(f"RMSE Bottom-Up: {rmse_bottomup:,.0f}")
print(f"RMSE MinTrace:  {rmse_mintrace:,.0f}")

Now we calculate the RMSE for each level

In [ ]:
levels = {
    'Total': ['TOTAL'],
    'Business Unit': [n for n in all_nodes_list if n.startswith('BU__')],
    'Segment': [n for n in all_nodes_list if n.startswith('SEG__')],
    'Subsegment': [n for n in all_nodes_list if n.startswith('SUBSEG__')],
}
 

In [ ]:
results_by_level = []
 
for level_name, nodes in levels.items():
    actual = Y_actual_pivot.loc[nodes].values.flatten()
    base = Y_hat_pivot.loc[nodes].values.flatten()
    bottomup = Y_bottomup_pivot.loc[nodes].values.flatten()
    mintrace = Y_reconciled_mintrace.loc[nodes].values.flatten()
    
    
    rmse_b = np.sqrt(mean_squared_error(actual, base))
    rmse_bu = np.sqrt(mean_squared_error(actual, bottomup))
    rmse_mt = np.sqrt(mean_squared_error(actual, mintrace))
    
    results_by_level.append({
        'Nível': level_name,
        'Base': rmse_b,
        'Bottom-Up': rmse_bu,
        'MinTrace': rmse_mt,
    })
    
    print(f"\n{level_name}:")
    print(f"  Base:       {rmse_b:>15,.0f}")
    print(f"  Bottom-Up:  {rmse_bu:>15,.0f}")
    print(f"  MinTrace:   {rmse_mt:>15,.0f}")
 
# DataFrame com resultados
results_df = pd.DataFrame(results_by_level)

## 6. ML Aproach

### 6.1. Feature Engineering

In [ ]:
df_fe = df_subseg.copy()

cols_to_drop = ['Biz_Desc', 'Business_Unit', 'Segment', 'Subsegment', 'Date', 'Orders']
df_fe = df_fe.drop(columns=[c for c in cols_to_drop if c in df_fe.columns])

df_fe = df_fe.rename(columns={
    'node': 'unique_id',
    'Revenue': 'y'
})

df_fe = df_fe.sort_values(['unique_id', 'Period'])

In [ ]:
def pre_fe(df):
    df_pre = df.copy()
    cols_to_drop = ['Biz_Desc', 'Business_Unit', 'Segment', 'Subsegment', 'Date', 'Orders']
    df_pre = df_pre.drop(columns=[c for c in cols_to_drop if c in df_pre.columns])

    df_pre = df_pre.rename(columns={
        'node': 'unique_id',
        'Revenue': 'y'
    })

    df_pre = df_pre.sort_values(['unique_id', 'Period'])

    return df_pre

#pre_fe(df_subseg)

#### 6.1.1 Lags

In [ ]:
def revenue_lags(df):
    df_lags = df.copy()

    lags = [1, 3, 6, 9]

    for lag in lags:
        df_lags[f'y_lag_{lag}'] = df_lags.groupby('unique_id')['y'].shift(lag)
    
    return df_lags

#revenue_lags(df_fe)

#### 6.1.2 Roling Features

In [ ]:
def revenue_roll(df):
    df_roll = df.copy()
    windows = [3, 6, 9]

    for w in windows:
        df_roll[f'y_roll_mean_{w}'] = df_roll.groupby('unique_id')['y'].shift(1).rolling(w).mean()
        #df_roll[f'y_roll_std_{w}'] = df_roll.groupby('unique_id')['y'].shift(1).rolling(w).std()
    
    return df_roll

#revenue_roll(df_fe)

#### 6.1.3 Time Features

In [ ]:
def time_features(df):
    df_tf = df.copy()
    
    df_tf['time_idx'] = df_tf.groupby('unique_id').cumcount()
    df_tf['month'] = df_tf['Period'] % 12

    return df_tf

#time_features(df_fe)

#### 6.1.4 Macro Features

In [ ]:
def macro_lags(df):
    df_macro_lags = df.copy()
    
    numeric_cols = df_macro_lags.select_dtypes(include=[np.number]).columns.tolist()
    macro_cols = [col for col in numeric_cols if col not in ['unique_id', 'Period', 'y']]

    macro_lags = [1, 3, 6]

    for col in macro_cols:
        if col.startswith('y_') or col in ['time_idx', 'month']:
            continue
    
        for lag in macro_lags:
            df_macro_lags[f'{col}_lag_{lag}'] = df_macro_lags.groupby('unique_id')[col].shift(lag)

    return df_macro_lags

#macro_lags(df_fe)

#### 6.1.5 Final Preprocessed Dataset

In [ ]:
df_model = macro_lags(time_features(revenue_roll(revenue_lags(df_fe)))).dropna().copy()

target = 'y'

features = [
    col for col in df_model.columns
    if col not in ['y', 'unique_id', 'Period']
]

### 6.2. Feature Selection

#### 6.2.1 Temporal Split (Expanding Window)

In [ ]:
expanding_folds = [
    {'train_end': 24, 'val_start': 25, 'val_end': 30},
    {'train_end': 30, 'val_start': 31, 'val_end': 36},
    {'train_end': 36, 'val_start': 37, 'val_end': 42},
]

In [ ]:
def get_expanding_fold(df, train_end, val_start, val_end):
    train_df = df[df['Period'] <= train_end].copy()
    val_df = df[(df['Period'] >= val_start) & (df['Period'] <= val_end)].copy()
    return train_df, val_df

In [ ]:
for i, fold in enumerate(expanding_folds, 1):
    train_df, val_df = get_expanding_fold(
        df_model,
        train_end=fold['train_end'],
        val_start=fold['val_start'],
        val_end=fold['val_end']
    )
    
    print(f"Fold {i}")
    print("Train shape:", train_df.shape)
    print("Val shape:", val_df.shape)
    print("-"*40)

In [ ]:
feature_cols = [c for c in df_model.columns if c not in ['unique_id', 'Period', 'y']]
print(f"Number of features: {len(feature_cols)}")

#### 6.2.2 XGBoost

In [ ]:
fold_feature_importances = []
fold_top_features = []
fold_rmse = []

TOP_N = 15

for i, fold in enumerate(expanding_folds, 1):
    train_df, val_df = get_expanding_fold(
        df_model,
        train_end=fold['train_end'],
        val_start=fold['val_start'],
        val_end=fold['val_end']
    )
    
    X_train = train_df[feature_cols]
    y_train = train_df['y']
    
    X_val = val_df[feature_cols]
    y_val = val_df['y']
    
    model = XGBRegressor(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='reg:squarederror',
        random_state=42,
        n_jobs=-1
    )
    
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    fold_rmse.append({
        'fold': i,
        'rmse': rmse
    })
    
    importance_df = pd.DataFrame({
        'feature': feature_cols,
        'importance': model.feature_importances_,
        'fold': i
    }).sort_values('importance', ascending=False)
    
    fold_feature_importances.append(importance_df)
    
    top_features = importance_df.head(TOP_N).copy()
    fold_top_features.append(top_features)
    
    print(f"Fold {i} RMSE: {rmse:,.2f}")
    print(top_features[['feature', 'importance']])
    print("-" * 60)

In [ ]:
all_top_features = pd.concat(fold_top_features, ignore_index=True)
feature_frequency = (
    all_top_features
    .groupby('feature')
    .size()
    .reset_index(name='n_folds_in_topN')
    .sort_values(['n_folds_in_topN', 'feature'], ascending=[False, True])
)

feature_frequency

In [ ]:
all_importances = pd.concat(fold_feature_importances, ignore_index=True)

mean_importance = (
    all_importances
    .groupby('feature', as_index=False)['importance']
    .mean()
    .sort_values('importance', ascending=False)
)

mean_importance.head(10)

#### 6.2.3 Lasso

In [ ]:
fold_lasso_coefs = []
fold_lasso_selected = []
fold_lasso_rmse = []

for i, fold in enumerate(expanding_folds, 1):
    train_df, val_df = get_expanding_fold(
        df_model,
        train_end=fold['train_end'],
        val_start=fold['val_start'],
        val_end=fold['val_end']
    )
    
    X_train = train_df[feature_cols]
    y_train = train_df['y']
    
    X_val = val_df[feature_cols]
    y_val = val_df['y']
    
    lasso_pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('lasso', LassoCV(
            cv=5,
            random_state=42,
            max_iter=20000,
            n_jobs=-1
        ))
    ])
    
    lasso_pipe.fit(X_train, y_train)
    
    y_pred = lasso_pipe.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    
    fold_lasso_rmse.append({
        'fold': i,
        'rmse': rmse,
        'alpha': lasso_pipe.named_steps['lasso'].alpha_
    })
    
    coefs = lasso_pipe.named_steps['lasso'].coef_
    
    coef_df = pd.DataFrame({
        'feature': feature_cols,
        'coef': coefs,
        'abs_coef': np.abs(coefs),
        'fold': i
    }).sort_values('abs_coef', ascending=False)
    
    fold_lasso_coefs.append(coef_df)
    
    selected_df = coef_df[coef_df['coef'] != 0].copy()
    fold_lasso_selected.append(selected_df)
    
    print(f"Fold {i} RMSE: {rmse:,.2f}")
    print(f"Selected features: {len(selected_df)}")
    print(f"Chosen alpha: {lasso_pipe.named_steps['lasso'].alpha_}")
    print(selected_df[['feature', 'coef', 'abs_coef']].sort_values(by = 'abs_coef', ascending=False).head(10))
    print("-" * 60)

#### 6.2.4 Final Insights

In [ ]:
# Frequência do Lasso: quantos folds cada feature foi seleccionada (coef != 0)
all_lasso_selected = pd.concat(fold_lasso_selected, ignore_index=True)

lasso_frequency = (
    all_lasso_selected
    .groupby('feature')
    .agg(
        lasso_folds=('fold', 'count'),
        mean_abs_coef=('abs_coef', 'mean')
    )
    .reset_index()
    .sort_values(['lasso_folds', 'mean_abs_coef'], ascending=[False, False])
)

In [ ]:
feature_selection_summary = mean_importance.merge(
    feature_frequency,
    on='feature',
    how='left'
).fillna({'n_folds_in_topN': 0})

feature_selection_summary = feature_selection_summary.sort_values(
    ['n_folds_in_topN', 'importance'],
    ascending=[False, False]
).reset_index(drop=True)

In [ ]:
# Juntar XGBoost + Lasso numa tabela combinada
combined = (
    feature_frequency
    .rename(columns={'n_folds_in_topN': 'xgb_folds'})
    .merge(lasso_frequency[['feature', 'lasso_folds', 'mean_abs_coef']],
           on='feature', how='outer')
    .fillna(0)
)
combined['xgb_folds']   = combined['xgb_folds'].astype(int)
combined['lasso_folds'] = combined['lasso_folds'].astype(int)
combined['total_score'] = combined['xgb_folds'] + combined['lasso_folds']
combined = combined.sort_values('total_score', ascending=False).reset_index(drop=True)

In [ ]:
# Visualização
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# XGBoost
top_xgb = combined.nlargest(20, 'xgb_folds')[['feature', 'xgb_folds']].iloc[::-1]
axes[0].barh(top_xgb['feature'], top_xgb['xgb_folds'], color='steelblue')
axes[0].axvline(x=2, color='red', linestyle='--', label='threshold >= 2')
axes[0].set_title('XGBoost — nº de folds no Top 15')
axes[0].set_xlabel('Folds')
axes[0].legend()

# Lasso
top_lasso = combined.nlargest(20, 'lasso_folds')[['feature', 'lasso_folds']].iloc[::-1]
axes[1].barh(top_lasso['feature'], top_lasso['lasso_folds'], color='darkorange')
axes[1].axvline(x=2, color='red', linestyle='--', label='threshold >= 2')
axes[1].set_title('Lasso — nº de folds seleccionada')
axes[1].set_xlabel('Folds')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
final_features = feature_selection_summary.head(10)['feature'].tolist()
#lasso_selected_features = combined[combined['lasso_folds'] >= 2]
final_features

## 7. Modeling

In [ ]:
df_subseg_final = df_model.copy()
df_seg_final = macro_lags(time_features(revenue_roll(revenue_lags(pre_fe(df_seg))))).dropna().copy()
df_bu_final = macro_lags(time_features(revenue_roll(revenue_lags(pre_fe(df_bu))))).dropna().copy()

In [ ]:
df_subseg_ets = df_subseg.rename(columns={
    'node': 'unique_id',
    'Revenue': 'y',
    'Period': 'ds'
})[['unique_id', 'ds', 'y']].copy()

df_seg_ets = df_seg.rename(columns={
    'node': 'unique_id',
    'Revenue': 'y',
    'Period': 'ds'
})[['unique_id', 'ds', 'y']].copy()

df_bu_ets = df_bu.rename(columns={
    'node': 'unique_id',
    'Revenue': 'y',
    'Period': 'ds'
})[['unique_id', 'ds', 'y']].copy()

In [ ]:
MIN_PERIOD = 10

df_subseg_ets = df_subseg_ets[df_subseg_ets['ds'] >= MIN_PERIOD].copy()
df_seg_ets = df_seg_ets[df_seg_ets['ds'] >= MIN_PERIOD].copy()
df_bu_ets = df_bu_ets[df_bu_ets['ds'] >= MIN_PERIOD].copy()

print("Subsegment series:", df_subseg_ets['unique_id'].nunique())
print("Segment series:", df_seg_ets['unique_id'].nunique())
print("BU series:", df_bu_ets['unique_id'].nunique())

In [ ]:
'''prophet_features = [
    'Japan_Steel_Production_lag_1',
    'United_States_Industrial_Production_Mom',
    'United_States_Industrial_Production_Mom_lag_3',
    'China_GDP_from_Manufacturing_lag_1'
]'''

prophet_features = [
    'Japan_Steel_Production',
    'United_States_Inflation_Rate',
    'China_Steel_Production',
    'Japan_Exports',
    'China_GDP_from_Construction']

df_subseg_prophet = df_subseg.rename(columns={
    'node': 'unique_id',
    'Revenue': 'y',
    'Period': 'period_num'
})[['unique_id', 'period_num', 'y'] + prophet_features].copy()

df_seg_prophet = df_seg.rename(columns={
    'node': 'unique_id',
    'Revenue': 'y',
    'Period': 'period_num'
})[['unique_id', 'period_num', 'y'] + prophet_features].copy()

df_bu_prophet = df_bu.rename(columns={
    'node': 'unique_id',
    'Revenue': 'y',
    'Period': 'period_num'
})[['unique_id', 'period_num', 'y'] + prophet_features].copy()

df_subseg_prophet = df_subseg_prophet[df_subseg_prophet['period_num'] >= 10].copy()
df_seg_prophet = df_seg_prophet[df_seg_prophet['period_num'] >= 10].copy()
df_bu_prophet = df_bu_prophet[df_bu_prophet['period_num'] >= 10].copy()

start_date = pd.Timestamp('2020-01-01')
df_seg_prophet['ds'] = df_seg_prophet['period_num'].apply(
    lambda x: start_date + pd.DateOffset(months=int(x)-1)
)
df_subseg_prophet['ds'] = df_subseg_prophet['period_num'].apply(
    lambda x: start_date + pd.DateOffset(months=int(x)-1)
)
df_bu_prophet['ds'] = df_bu_prophet['period_num'].apply(
    lambda x: start_date + pd.DateOffset(months=int(x)-1)
)

### 7.1 Baseline Models

#### 7.1.1 Subsegment Level

##### 7.1.1.1 XGBoost

In [ ]:
xgb_results_sub = []

for i, fold in enumerate(expanding_folds, 1):
    train_df, val_df = get_expanding_fold(
        df_subseg_final,
        train_end=fold['train_end'],
        val_start=fold['val_start'],
        val_end=fold['val_end']
    )
    
    X_train = train_df[final_features]
    y_train = train_df['y']
    
    X_val = val_df[final_features]
    y_val = val_df['y']
    
    model = XGBRegressor(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='reg:squarederror',
        random_state=42,
        n_jobs=-1
    )
    
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    
    xgb_results_sub.append(rmse)
    
    print(f"XGBoost Fold {i} RMSE: {rmse:,.2f}")

print("XGBoost Mean RMSE:", np.mean(xgb_results_sub))

##### 7.1.1.2 LightGBM

In [ ]:
lgbm_results_sub = []

for i, fold in enumerate(expanding_folds, 1):
    train_df, val_df = get_expanding_fold(
        df_subseg_final,
        train_end=fold['train_end'],
        val_start=fold['val_start'],
        val_end=fold['val_end']
    )
    
    X_train = train_df[final_features]
    y_train = train_df['y']
    
    X_val = val_df[final_features]
    y_val = val_df['y']
    
    model = lgb.LGBMRegressor(
        n_estimators=200,
        learning_rate=0.1,
        num_leaves=31,
        max_depth=-1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )
    
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    lgbm_results_sub.append(rmse)
    
    print(f"LightGBM Fold {i} RMSE: {rmse:,.2f}")

print(lgbm_results_sub)
print("LightGBM Mean RMSE:", np.mean(lgbm_results_sub))

##### 7.1.1.3 Prophet

In [ ]:
prophet_results_sub = []
prophet_fold_details = []

for i, fold in enumerate(expanding_folds, 1):
    train_df, val_df = get_expanding_fold(
        df_subseg_final,
        train_end=fold['train_end'],
        val_start=fold['val_start'],
        val_end=fold['val_end']
    )
    
    preds = []
    actuals = []
    valid_series = 0
    failed_series = []
    
    for uid in train_df['unique_id'].unique():
        train_series = train_df[train_df['unique_id'] == uid].copy()
        val_series = val_df[val_df['unique_id'] == uid].copy()
        
        # proteger caso alguma série não tenha validation
        if len(val_series) == 0:
            continue
        
        # formato Prophet
        df_train = train_series[['Period', 'y']].rename(columns={'Period': 'ds', 'y': 'y'}).copy()
        df_val = val_series[['Period', 'y']].rename(columns={'Period': 'ds', 'y': 'y'}).copy()
        
        # Prophet gosta de ds como datetime
        # se Period já for datetime, isto não estraga
        df_train['ds'] = df_train['ds'].apply(lambda p: pd.Timestamp('2019-01-01') + pd.DateOffset(months=int(p)-1))
        df_val['ds'] = df_val['ds'].apply(lambda p: pd.Timestamp('2019-01-01') + pd.DateOffset(months=int(p)-1))
        
        # adicionar regressoras
        for col in prophet_features:
            df_train[col] = train_series[col].values
            df_val[col] = val_series[col].values
        
        try:
            model = Prophet(
                yearly_seasonality=True,
                weekly_seasonality=False,
                daily_seasonality=False
            )
            
            for col in prophet_features:
                model.add_regressor(col)
            
            model.fit(df_train)
            
            forecast = model.predict(df_val[['ds'] + prophet_features])
            
            preds.extend(forecast['yhat'].values)
            actuals.extend(df_val['y'].values)
            valid_series += 1
            
        except Exception as e:
            failed_series.append((uid, str(e)))
            continue
    
    if len(preds) == 0:
        print(f"Prophet Fold {i}: 0 séries válidas")
        prophet_results_sub.append(np.nan)
        prophet_fold_details.append({
            'fold': i,
            'rmse': np.nan,
            'valid_series': 0,
            'failed_series': len(failed_series)
        })
        continue
    
    rmse = np.sqrt(mean_squared_error(actuals, preds))
    prophet_results_sub.append(rmse)
    
    prophet_fold_details.append({
        'fold': i,
        'rmse': rmse,
        'valid_series': valid_series,
        'failed_series': len(failed_series)
    })
    
    print(f"Prophet Fold {i} RMSE: {rmse:,.2f}")
    print(f"Séries válidas: {valid_series}")
    print(f"Séries falhadas: {len(failed_series)}")
    print("-" * 60)

print("Prophet Mean RMSE:", np.nanmean(prophet_results_sub))

In [ ]:
prophet_results_sub

##### 7.1.1.4 AutoETS

In [ ]:
def run_autoets(df_sf, expanding_folds, horizon=6, level_name=""):
    
    results = []
    
    for i, fold in enumerate(expanding_folds, 1):
        
        train_df = df_sf[df_sf['ds'] <= fold['train_end']].copy()
        val_df = df_sf[
            (df_sf['ds'] >= fold['val_start']) &
            (df_sf['ds'] <= fold['val_end'])
        ].copy()
        
        sf = StatsForecast(
            models=[AutoETS(season_length=12)],
            freq=1,
            n_jobs=-1
        )
        
        sf.fit(train_df)
        preds = sf.predict(h=horizon).reset_index()
        
        eval_df = val_df.merge(preds, on=['unique_id', 'ds'], how='inner')
        
        rmse = np.sqrt(mean_squared_error(eval_df['y'], eval_df['AutoETS']))
        results.append(rmse)
        
        print(f"AutoETS {level_name} Fold {i} RMSE: {rmse:,.2f}")
    
    print(f"AutoETS {level_name} Mean RMSE:", np.mean(results))
    
    return results

In [ ]:
autoets_subseg_results_sub = run_autoets(
    df_subseg_ets,
    expanding_folds,
    horizon=6,
    level_name = "Subsegment"
)

##### 7.1.1.5 ETS

In [ ]:
def run_ets_baseline(df_sf, expanding_folds, config, horizon=6, level_name=""):
    error, trend, seasonal = config
    
    results = []
    fold_details = []
    
    for i, fold in enumerate(expanding_folds, 1):
        
        train_df = df_sf[df_sf['ds'] <= fold['train_end']].copy()
        val_df = df_sf[
            (df_sf['ds'] >= fold['val_start']) &
            (df_sf['ds'] <= fold['val_end'])
        ].copy()
        
        sf = StatsForecast(
            models=[ETS(
                error=error,
                trend=trend,
                seasonal=seasonal,
                season_length=12
            )],
            freq=1,
            n_jobs=-1
        )
        
        try:
            sf.fit(train_df)
            preds = sf.predict(h=horizon).reset_index()
            
            eval_df = val_df.merge(preds, on=['unique_id', 'ds'], how='inner')
            
            rmse = np.sqrt(mean_squared_error(eval_df['y'], eval_df['ETS']))
            results.append(rmse)
            
            fold_details.append({
                'fold': i,
                'rmse': rmse,
                'n_obs': len(eval_df)
            })
            
            print(f"ETS {level_name} Fold {i} RMSE: {rmse:,.2f}")
        
        except Exception as e:
            results.append(np.nan)
            fold_details.append({
                'fold': i,
                'rmse': np.nan,
                'n_obs': 0
            })
            
            print(f"ETS {level_name} Fold {i} falhou: {e}")
    
    mean_rmse = np.nanmean(results)
    print(f"ETS {level_name} Mean RMSE: {mean_rmse:,.2f}")
    
    return results, pd.DataFrame(fold_details)

In [ ]:
#from statsforecast.models import ETS
#ets_subseg_results, ets_subseg_details = run_ets_baseline(
 #   df_subseg_ets,
  #  expanding_folds,
   # config=('add', 'add', 'add'),
    #horizon=6,
    #level_name="Subsegment"
#)

#### 7.1.2 Segment Level

##### 7.1.2.1 XGBoost

In [ ]:
xgb_results_seg = []

for i, fold in enumerate(expanding_folds, 1):
    train_df, val_df = get_expanding_fold(
        df_seg_final,
        train_end=fold['train_end'],
        val_start=fold['val_start'],
        val_end=fold['val_end']
    )
    
    X_train = train_df[final_features]
    y_train = train_df['y']
    
    X_val = val_df[final_features]
    y_val = val_df['y']
    
    model = XGBRegressor(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='reg:squarederror',
        random_state=42,
        n_jobs=-1
    )
    
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    
    xgb_results_seg.append(rmse)
    
    print(f"XGBoost Fold {i} RMSE: {rmse:,.2f}")

print("XGBoost Mean RMSE:", np.mean(xgb_results_seg))

##### 7.1.2.2 LightGBM

In [ ]:
lgbm_results_seg = []

for i, fold in enumerate(expanding_folds, 1):
    train_df, val_df = get_expanding_fold(
        df_seg_final,
        train_end=fold['train_end'],
        val_start=fold['val_start'],
        val_end=fold['val_end']
    )
    
    X_train = train_df[final_features]
    y_train = train_df['y']
    
    X_val = val_df[final_features]
    y_val = val_df['y']
    
    model = lgb.LGBMRegressor(
        n_estimators=200,
        learning_rate=0.1,
        num_leaves=31,
        max_depth=-1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )
    
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    lgbm_results_seg.append(rmse)
    
    print(f"LightGBM Fold {i} RMSE: {rmse:,.2f}")

print(lgbm_results_seg)
print("LightGBM Mean RMSE:", np.mean(lgbm_results_seg))

##### 7.1.2.3 Prophet

In [ ]:
prophet_results_seg = []
prophet_fold_details = []

for i, fold in enumerate(expanding_folds, 1):
    train_df, val_df = get_expanding_fold(
        df_seg_final,
        train_end=fold['train_end'],
        val_start=fold['val_start'],
        val_end=fold['val_end']
    )
    
    preds = []
    actuals = []
    valid_series = 0
    failed_series = []
    
    for uid in train_df['unique_id'].unique():
        train_series = train_df[train_df['unique_id'] == uid].copy()
        val_series = val_df[val_df['unique_id'] == uid].copy()
        
        # proteger caso alguma série não tenha validation
        if len(val_series) == 0:
            continue
        
        # formato Prophet
        df_train = train_series[['Period', 'y']].rename(columns={'Period': 'ds', 'y': 'y'}).copy()
        df_val = val_series[['Period', 'y']].rename(columns={'Period': 'ds', 'y': 'y'}).copy()
        
        # Prophet gosta de ds como datetime
        # se Period já for datetime, isto não estraga
        df_train['ds'] = df_train['ds'].apply(lambda p: pd.Timestamp('2019-01-01') + pd.DateOffset(months=int(p)-1))
        df_val['ds'] = df_val['ds'].apply(lambda p: pd.Timestamp('2019-01-01') + pd.DateOffset(months=int(p)-1))
        
        # adicionar regressoras
        for col in prophet_features:
            df_train[col] = train_series[col].values
            df_val[col] = val_series[col].values
        
        try:
            model = Prophet(
                yearly_seasonality=True,
                weekly_seasonality=False,
                daily_seasonality=False
            )
            
            for col in prophet_features:
                model.add_regressor(col)
            
            model.fit(df_train)
            
            forecast = model.predict(df_val[['ds'] + prophet_features])
            
            preds.extend(forecast['yhat'].values)
            actuals.extend(df_val['y'].values)
            valid_series += 1
            
        except Exception as e:
            failed_series.append((uid, str(e)))
            continue
    
    if len(preds) == 0:
        print(f"Prophet Fold {i}: 0 séries válidas")
        prophet_results_seg.append(np.nan)
        prophet_fold_details.append({
            'fold': i,
            'rmse': np.nan,
            'valid_series': 0,
            'failed_series': len(failed_series)
        })
        continue
    
    rmse = np.sqrt(mean_squared_error(actuals, preds))
    prophet_results_seg.append(rmse)
    
    prophet_fold_details.append({
        'fold': i,
        'rmse': rmse,
        'valid_series': valid_series,
        'failed_series': len(failed_series)
    })
    
    print(f"Prophet Fold {i} RMSE: {rmse:,.2f}")
    print(f"Séries válidas: {valid_series}")
    print(f"Séries falhadas: {len(failed_series)}")
    print("-" * 60)

print("Prophet Mean RMSE:", np.nanmean(prophet_results_seg))

In [ ]:
prophet_results_seg

##### 7.1.2.4 AutoETS

In [ ]:
autoets_subseg_results_seg = run_autoets(
    df_seg_ets,
    expanding_folds,
    horizon=6,
    level_name = "Segment"
)

#### 7.1.3 Business Unit Level

##### 7.1.3.1 XGBoost

In [ ]:
xgb_results_bu = []

for i, fold in enumerate(expanding_folds, 1):
    train_df, val_df = get_expanding_fold(
        df_bu_final,
        train_end=fold['train_end'],
        val_start=fold['val_start'],
        val_end=fold['val_end']
    )
    
    X_train = train_df[final_features]
    y_train = train_df['y']
    
    X_val = val_df[final_features]
    y_val = val_df['y']
    
    model = XGBRegressor(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='reg:squarederror',
        random_state=42,
        n_jobs=-1
    )
    
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    
    xgb_results_bu.append(rmse)
    
    print(f"XGBoost Fold {i} RMSE: {rmse:,.2f}")

print("XGBoost Mean RMSE:", np.mean(xgb_results_bu))

##### 7.1.3.2 LightGBM

In [ ]:
lgbm_results_bu = []

for i, fold in enumerate(expanding_folds, 1):
    train_df, val_df = get_expanding_fold(
        df_bu_final,
        train_end=fold['train_end'],
        val_start=fold['val_start'],
        val_end=fold['val_end']
    )
    
    X_train = train_df[final_features]
    y_train = train_df['y']
    
    X_val = val_df[final_features]
    y_val = val_df['y']
    
    model = lgb.LGBMRegressor(
        n_estimators=200,
        learning_rate=0.1,
        num_leaves=31,
        max_depth=-1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )
    
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    lgbm_results_bu.append(rmse)
    
    print(f"LightGBM Fold {i} RMSE: {rmse:,.2f}")

print(lgbm_results_bu)
print("LightGBM Mean RMSE:", np.mean(lgbm_results_bu))

##### 7.1.3.3 Prophet

In [ ]:
prophet_results_bu = []
prophet_fold_details = []

for i, fold in enumerate(expanding_folds, 1):
    train_df, val_df = get_expanding_fold(
        df_bu_final,
        train_end=fold['train_end'],
        val_start=fold['val_start'],
        val_end=fold['val_end']
    )
    
    preds = []
    actuals = []
    valid_series = 0
    failed_series = []
    
    for uid in train_df['unique_id'].unique():
        train_series = train_df[train_df['unique_id'] == uid].copy()
        val_series = val_df[val_df['unique_id'] == uid].copy()
        
        # proteger caso alguma série não tenha validation
        if len(val_series) == 0:
            continue
        
        # formato Prophet
        df_train = train_series[['Period', 'y']].rename(columns={'Period': 'ds', 'y': 'y'}).copy()
        df_val = val_series[['Period', 'y']].rename(columns={'Period': 'ds', 'y': 'y'}).copy()
        
        # Prophet gosta de ds como datetime
        # se Period já for datetime, isto não estraga
        df_train['ds'] = df_train['ds'].apply(lambda p: pd.Timestamp('2019-01-01') + pd.DateOffset(months=int(p)-1))
        df_val['ds'] = df_val['ds'].apply(lambda p: pd.Timestamp('2019-01-01') + pd.DateOffset(months=int(p)-1))
        
        # adicionar regressoras
        for col in prophet_features:
            df_train[col] = train_series[col].values
            df_val[col] = val_series[col].values
        
        try:
            model = Prophet(
                yearly_seasonality=True,
                weekly_seasonality=False,
                daily_seasonality=False
            )
            
            for col in prophet_features:
                model.add_regressor(col)
            
            model.fit(df_train)
            
            forecast = model.predict(df_val[['ds'] + prophet_features])
            
            preds.extend(forecast['yhat'].values)
            actuals.extend(df_val['y'].values)
            valid_series += 1
            
        except Exception as e:
            failed_series.append((uid, str(e)))
            continue
    
    if len(preds) == 0:
        print(f"Prophet Fold {i}: 0 séries válidas")
        prophet_results_bu.append(np.nan)
        prophet_fold_details.append({
            'fold': i,
            'rmse': np.nan,
            'valid_series': 0,
            'failed_series': len(failed_series)
        })
        continue
    
    rmse = np.sqrt(mean_squared_error(actuals, preds))
    prophet_results_bu.append(rmse)
    
    prophet_fold_details.append({
        'fold': i,
        'rmse': rmse,
        'valid_series': valid_series,
        'failed_series': len(failed_series)
    })
    
    print(f"Prophet Fold {i} RMSE: {rmse:,.2f}")
    print(f"Séries válidas: {valid_series}")
    print(f"Séries falhadas: {len(failed_series)}")
    print("-" * 60)

print("Prophet Mean RMSE:", np.nanmean(prophet_results_bu))

In [ ]:
prophet_results_bu

##### 7.1.3.4 AutoETS

In [ ]:
autoets_subseg_results_bu = run_autoets(
    df_bu_ets,
    expanding_folds,
    horizon=6,
    level_name = "BU"
)

#### 7.1.4 Comparison

##### 7.1.4.1 Subsegment Level

In [ ]:
print("XGBoost Mean RMSE: ", np.mean(xgb_results_sub))
print("LightGBM Mean RMSE: ", np.mean(lgbm_results_sub))
print("Prophet Mean RMSE: ", np.mean(prophet_results_sub))
print("AutoETS Mean RMSE: ", np.mean(autoets_subseg_results_sub))

##### 7.1.4.2 Segment Level

In [ ]:
print("XGBoost Mean RMSE: ", np.mean(xgb_results_seg))
print("LightGBM Mean RMSE: ", np.mean(lgbm_results_seg))
print("Prophet Mean RMSE: ", np.mean(prophet_results_seg))
print("AutoETS Mean RMSE: ", np.mean(autoets_subseg_results_seg))

##### 7.1.4.3 Business Unit Level

In [ ]:
print("XGBoost Mean RMSE: ", np.mean(xgb_results_bu))
print("LightGBM Mean RMSE: ", np.mean(lgbm_results_bu))
print("Prophet Mean RMSE: ", np.mean(prophet_results_bu))
print("AutoETS Mean RMSE: ", np.mean(autoets_subseg_results_bu))

In [ ]:
oi

### 7.2 Tuning

#### 7.2.1 Subsegment Level

##### 7.2.1.1 XGBoost

In [ ]:
n_trials = 20  # número de combinações aleatórias a testar — aumentar para explorar mais
xgb_random_results = []

for i in range(n_trials):
    params = {
        'n_estimators'    : random.choice([150, 200, 250, 300, 350]),  # nº de árvores — mais = mais lento mas potencialmente melhor
        'max_depth'       : random.choice([3, 4, 5, 6]),               # profundidade de cada árvore — mais alto = mais overfitting
        'learning_rate'   : random.choice([0.03, 0.05, 0.07, 0.1]),   # tamanho do passo — mais baixo precisa de mais árvores
        'subsample'       : random.choice([0.7, 0.8, 0.9]),            # % de linhas usadas por árvore — ajuda a regularizar
        'colsample_bytree': random.choice([0.7, 0.8, 0.9]),            # % de features usadas por árvore — ajuda a regularizar
        'min_child_weight': random.choice([1, 3, 5, 7]),               # mínimo de peso por folha — mais alto = menos overfitting
    }

    fold_rmses = []
    for fold in expanding_folds:
        train_df, val_df = get_expanding_fold(
            df_subseg_final, fold['train_end'], fold['val_start'], fold['val_end']
        )
        model = XGBRegressor(objective='reg:squarederror', random_state=42, n_jobs=-1, **params)
        model.fit(train_df[final_features], train_df['y'])
        y_pred = model.predict(val_df[final_features])
        fold_rmses.append(np.sqrt(mean_squared_error(val_df['y'], y_pred)))

    result = params.copy()
    result['mean_rmse'] = np.mean(fold_rmses)
    result['std_rmse']  = np.std(fold_rmses)
    xgb_random_results.append(result)

    print(f"Trial {i+1}/{n_trials} | Mean RMSE: {result['mean_rmse']:,.2f} | {params}")
    print("-" * 60)

xgb_random_subseg = pd.DataFrame(xgb_random_results).sort_values('mean_rmse').reset_index(drop=True)
xgb_random_subseg.head(10)

##### 7.2.1.2 LightGBM

In [ ]:
n_trials = 20  # número de combinações aleatórias a testar — aumentar para explorar mais
lgbm_random_results = []

for i in range(n_trials):
    params = {
        'n_estimators'     : random.choice([150, 200, 250, 300, 350]),  # nº de árvores — mais = mais lento mas potencialmente melhor
        'num_leaves'       : random.choice([15, 31, 50, 63]),           # complexidade de cada árvore — valores altos = mais overfitting
        'learning_rate'    : random.choice([0.03, 0.05, 0.07, 0.1]),   # tamanho do passo — mais baixo precisa de mais árvores
        'subsample'        : random.choice([0.7, 0.8, 0.9]),            # % de linhas usadas por árvore — ajuda a regularizar
        'colsample_bytree' : random.choice([0.7, 0.8, 0.9]),            # % de features usadas por árvore — ajuda a regularizar
        'min_child_samples': random.choice([5, 10, 20, 30]),            # mínimo de obs por folha — mais alto = menos overfitting
        'reg_alpha'        : random.choice([0, 0.01, 0.1]),             # regularização L1 — penaliza coeficientes pequenos
        'reg_lambda'       : random.choice([0, 0.01, 0.1]),             # regularização L2 — penaliza coeficientes grandes
    }

    fold_rmses = []
    for fold in expanding_folds:
        train_df, val_df = get_expanding_fold(
            df_subseg_final, fold['train_end'], fold['val_start'], fold['val_end']
        )
        model = lgb.LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1, **params)
        model.fit(train_df[final_features], train_df['y'])
        y_pred = model.predict(val_df[final_features])
        fold_rmses.append(np.sqrt(mean_squared_error(val_df['y'], y_pred)))

    result = params.copy()
    result['mean_rmse'] = np.mean(fold_rmses)
    result['std_rmse']  = np.std(fold_rmses)
    lgbm_random_results.append(result)

    print(f"Trial {i+1}/{n_trials} | Mean RMSE: {result['mean_rmse']:,.2f} | {params}")
    print("-" * 60)

lgbm_random_subseg = pd.DataFrame(lgbm_random_results).sort_values('mean_rmse').reset_index(drop=True)
lgbm_random_subseg.head(10)


##### 7.2.1.3 Prophet

In [ ]:
logging.getLogger('prophet').setLevel(logging.WARNING)  # silenciar logs do Prophet

n_trials = 20  # número de combinações aleatórias a testar
prophet_random_results = []

for i in range(n_trials):
    params = {
        'changepoint_prior_scale' : random.choice([0.001, 0.01, 0.05, 0.1, 0.3]),   # flexibilidade da tendência — mais alto = tendência mais volátil
        'seasonality_prior_scale' : random.choice([0.01, 0.1, 1.0, 5.0, 10.0]),     # força da sazonalidade — mais alto = sazonalidade mais pronunciada
        'seasonality_mode'        : random.choice(['additive', 'multiplicative']),   # additive: sazonalidade constante; multiplicative: sazonalidade proporcional ao nível
        'changepoint_range'       : random.choice([0.8, 0.85, 0.9]),                 # % do histórico onde podem ocorrer mudanças de tendência — mais alto = mais flexível no fim
    }

    fold_rmses = []
    for fold in expanding_folds:
        train_df, val_df = get_expanding_fold(
            df_subseg_final, fold['train_end'], fold['val_start'], fold['val_end']
        )

        preds, actuals = [], []
        for uid in train_df['unique_id'].unique():
            ts = train_df[train_df['unique_id'] == uid].copy()
            vs = val_df[val_df['unique_id'] == uid].copy()
            if len(vs) == 0:
                continue

            df_tr = ts[['Period','y']].rename(columns={'Period':'ds'})
            df_vl = vs[['Period','y']].rename(columns={'Period':'ds'})
            # converter Period (inteiro) em datas reais — origem = Jan 2019, 1 período = 1 mês
            df_tr['ds'] = df_tr['ds'].apply(lambda p: pd.Timestamp('2019-01-01') + pd.DateOffset(months=int(p)-1))
            df_vl['ds'] = df_vl['ds'].apply(lambda p: pd.Timestamp('2019-01-01') + pd.DateOffset(months=int(p)-1))
            for col in prophet_features:
                df_tr[col] = ts[col].values
                df_vl[col] = vs[col].values

            try:
                m = Prophet(
                    yearly_seasonality=True,   # activar sazonalidade anual — faz sentido com dados mensais
                    weekly_seasonality=False,  # desactivado — dados mensais não têm padrão semanal
                    daily_seasonality=False,   # desactivado — dados mensais não têm padrão diário
                    **params
                )
                for col in prophet_features:
                    m.add_regressor(col)  # adicionar macro features como regressores externos
                m.fit(df_tr)
                fc = m.predict(df_vl[['ds'] + prophet_features])
                preds.extend(fc['yhat'].values)
                actuals.extend(df_vl['y'].values)
            except:
                continue

        if preds:
            fold_rmses.append(np.sqrt(mean_squared_error(actuals, preds)))

    result = params.copy()
    result['mean_rmse'] = np.mean(fold_rmses) if fold_rmses else np.nan
    result['std_rmse']  = np.std(fold_rmses)  if fold_rmses else np.nan
    prophet_random_results.append(result)

    print(f"Trial {i+1}/{n_trials} | Mean RMSE: {result['mean_rmse']:,.2f} | {params}")
    print("-" * 60)

prophet_random_subseg = pd.DataFrame(prophet_random_results).sort_values('mean_rmse').reset_index(drop=True)
prophet_random_subseg.head(10)

##### ETS

In [ ]:
"""# Parâmetros possíveis do ETS — cada modelo é definido por 3 letras (Erro + Tendência + Sazonalidade):
#   Erro:       A = aditivo,        M = multiplicativo
#   Tendência:  A = aditiva,        N = sem tendência
#   Sazonalidade: A = aditiva,      M = multiplicativa,   N = sem sazonalidade
# Exemplo: 'AAN' = erro aditivo, tendência aditiva, sem sazonalidade
error_types  = ['A', 'M']
trend_types  = ['A', 'N']
season_types = ['A', 'M', 'N']

n_trials = 20
ets_random_results = []

for i in range(n_trials):
    error  = random.choice(error_types)
    trend  = random.choice(trend_types)
    season = random.choice(season_types)
    # damped amorte a tendência ao longo do tempo — só faz sentido se houver tendência
    damped = random.choice([True, False]) if trend != 'N' else False

    params = {'model': error + trend + season, 'damped': damped}

    fold_rmses = []
    for fold in expanding_folds:
        train_df, val_df = get_expanding_fold(
            df_subseg_final, fold['train_end'], fold['val_start'], fold['val_end']
        )

        # ETS usa apenas unique_id, ds, y — sem features externas
        Y_tr  = train_df[['unique_id', 'Period', 'y']].rename(columns={'Period': 'ds'})
        Y_val = val_df[['unique_id', 'Period', 'y']].rename(columns={'Period': 'ds'})

        try:
            sf = StatsForecast(
                models=[ETS(season_length=12, **params)],
                freq=1, n_jobs=-1
            )
            sf.fit(Y_tr)
            preds  = sf.predict(h=int(Y_val['ds'].nunique())).reset_index()
            col    = [c for c in preds.columns if c not in ['unique_id', 'ds']][0]
            merged = Y_val.merge(preds, on=['unique_id', 'ds'], how='left').dropna(subset=[col])
            fold_rmses.append(np.sqrt(mean_squared_error(merged['y'], merged[col])))
        except Exception:
            continue  # alguns modelos falham com séries muito curtas ou com valores negativos

    result = {**params, 'mean_rmse': np.mean(fold_rmses) if fold_rmses else np.nan}
    ets_random_results.append(result)
    print(f"Trial {i+1}/{n_trials} | RMSE: {result['mean_rmse']:,.2f} | {params}")

ets_random_subseg = pd.DataFrame(ets_random_results).sort_values('mean_rmse').reset_index(drop=True)
ets_random_subseg.head(10)"""

##### 7.2.1.5 Comparison

In [ ]:
print("=== Melhores Parâmetros ===")

print("\nXGBoost:")
print(xgb_random_subseg.iloc[0][['n_estimators','max_depth','learning_rate',
                               'subsample','colsample_bytree','min_child_weight']].to_dict())

print("\nLightGBM:")
print(lgbm_random_subseg.iloc[0][['n_estimators','num_leaves','learning_rate',
                                'subsample','colsample_bytree','min_child_samples',
                                'reg_alpha','reg_lambda']].to_dict())

print("\nProphet:")
print(prophet_random_subseg.iloc[0][['changepoint_prior_scale','seasonality_prior_scale',
                                  'seasonality_mode','changepoint_range']].to_dict())

comparison = pd.DataFrame({
    'Model': ['XGBoost', 'LightGBM', 'Prophet'],
    'Mean RMSE': [
        xgb_random_subseg.iloc[0]['mean_rmse'],
        lgbm_random_subseg.iloc[0]['mean_rmse'],
        prophet_random_subseg.iloc[0]['mean_rmse']
    ]
}).sort_values('Mean RMSE').reset_index(drop=True)

print("\n=== RMSE por Modelo ===")
print(comparison.to_string(index=False))

#### 7.2.2 Segment Level

##### 7.2.2.1 XGBoost

In [ ]:
n_trials = 20  # número de combinações aleatórias a testar — aumentar para explorar mais
xgb_random_results = []

for i in range(n_trials):
    params = {
        'n_estimators'    : random.choice([150, 200, 250, 300, 350]),  # nº de árvores — mais = mais lento mas potencialmente melhor
        'max_depth'       : random.choice([3, 4, 5, 6]),               # profundidade de cada árvore — mais alto = mais overfitting
        'learning_rate'   : random.choice([0.03, 0.05, 0.07, 0.1]),   # tamanho do passo — mais baixo precisa de mais árvores
        'subsample'       : random.choice([0.7, 0.8, 0.9]),            # % de linhas usadas por árvore — ajuda a regularizar
        'colsample_bytree': random.choice([0.7, 0.8, 0.9]),            # % de features usadas por árvore — ajuda a regularizar
        'min_child_weight': random.choice([1, 3, 5, 7]),               # mínimo de peso por folha — mais alto = menos overfitting
    }

    fold_rmses = []
    for fold in expanding_folds:
        train_df, val_df = get_expanding_fold(
            df_seg_final, fold['train_end'], fold['val_start'], fold['val_end']
        )
        model = XGBRegressor(objective='reg:squarederror', random_state=42, n_jobs=-1, **params)
        model.fit(train_df[final_features], train_df['y'])
        y_pred = model.predict(val_df[final_features])
        fold_rmses.append(np.sqrt(mean_squared_error(val_df['y'], y_pred)))

    result = params.copy()
    result['mean_rmse'] = np.mean(fold_rmses)
    result['std_rmse']  = np.std(fold_rmses)
    xgb_random_results.append(result)

    print(f"Trial {i+1}/{n_trials} | Mean RMSE: {result['mean_rmse']:,.2f} | {params}")
    print("-" * 60)

xgb_random_seg = pd.DataFrame(xgb_random_results).sort_values('mean_rmse').reset_index(drop=True)
xgb_random_seg.head(10)

##### 7.2.2.2 LightGBM

In [ ]:
n_trials = 20  # número de combinações aleatórias a testar — aumentar para explorar mais
lgbm_random_results = []

for i in range(n_trials):
    params = {
        'n_estimators'     : random.choice([150, 200, 250, 300, 350]),  # nº de árvores — mais = mais lento mas potencialmente melhor
        'num_leaves'       : random.choice([15, 31, 50, 63]),           # complexidade de cada árvore — valores altos = mais overfitting
        'learning_rate'    : random.choice([0.03, 0.05, 0.07, 0.1]),   # tamanho do passo — mais baixo precisa de mais árvores
        'subsample'        : random.choice([0.7, 0.8, 0.9]),            # % de linhas usadas por árvore — ajuda a regularizar
        'colsample_bytree' : random.choice([0.7, 0.8, 0.9]),            # % de features usadas por árvore — ajuda a regularizar
        'min_child_samples': random.choice([5, 10, 20, 30]),            # mínimo de obs por folha — mais alto = menos overfitting
        'reg_alpha'        : random.choice([0, 0.01, 0.1]),             # regularização L1 — penaliza coeficientes pequenos
        'reg_lambda'       : random.choice([0, 0.01, 0.1]),             # regularização L2 — penaliza coeficientes grandes
    }

    fold_rmses = []
    for fold in expanding_folds:
        train_df, val_df = get_expanding_fold(
            df_seg_final, fold['train_end'], fold['val_start'], fold['val_end']
        )
        model = lgb.LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1, **params)
        model.fit(train_df[final_features], train_df['y'])
        y_pred = model.predict(val_df[final_features])
        fold_rmses.append(np.sqrt(mean_squared_error(val_df['y'], y_pred)))

    result = params.copy()
    result['mean_rmse'] = np.mean(fold_rmses)
    result['std_rmse']  = np.std(fold_rmses)
    lgbm_random_results.append(result)

    print(f"Trial {i+1}/{n_trials} | Mean RMSE: {result['mean_rmse']:,.2f} | {params}")
    print("-" * 60)

lgbm_random_seg = pd.DataFrame(lgbm_random_results).sort_values('mean_rmse').reset_index(drop=True)
lgbm_random_seg.head(10)


##### 7.2.2.3 Prophet

In [ ]:
logging.getLogger('prophet').setLevel(logging.WARNING)  # silenciar logs do Prophet

n_trials = 20  # número de combinações aleatórias a testar
prophet_random_results = []

for i in range(n_trials):
    params = {
        'changepoint_prior_scale' : random.choice([0.001, 0.01, 0.05, 0.1, 0.3]),   # flexibilidade da tendência — mais alto = tendência mais volátil
        'seasonality_prior_scale' : random.choice([0.01, 0.1, 1.0, 5.0, 10.0]),     # força da sazonalidade — mais alto = sazonalidade mais pronunciada
        'seasonality_mode'        : random.choice(['additive', 'multiplicative']),   # additive: sazonalidade constante; multiplicative: sazonalidade proporcional ao nível
        'changepoint_range'       : random.choice([0.8, 0.85, 0.9]),                 # % do histórico onde podem ocorrer mudanças de tendência — mais alto = mais flexível no fim
    }

    fold_rmses = []
    for fold in expanding_folds:
        train_df, val_df = get_expanding_fold(
            df_seg_final, fold['train_end'], fold['val_start'], fold['val_end']
        )

        preds, actuals = [], []
        for uid in train_df['unique_id'].unique():
            ts = train_df[train_df['unique_id'] == uid].copy()
            vs = val_df[val_df['unique_id'] == uid].copy()
            if len(vs) == 0:
                continue

            df_tr = ts[['Period','y']].rename(columns={'Period':'ds'})
            df_vl = vs[['Period','y']].rename(columns={'Period':'ds'})
            # converter Period (inteiro) em datas reais — origem = Jan 2019, 1 período = 1 mês
            df_tr['ds'] = df_tr['ds'].apply(lambda p: pd.Timestamp('2019-01-01') + pd.DateOffset(months=int(p)-1))
            df_vl['ds'] = df_vl['ds'].apply(lambda p: pd.Timestamp('2019-01-01') + pd.DateOffset(months=int(p)-1))
            for col in prophet_features:
                df_tr[col] = ts[col].values
                df_vl[col] = vs[col].values

            try:
                m = Prophet(
                    yearly_seasonality=True,   # activar sazonalidade anual — faz sentido com dados mensais
                    weekly_seasonality=False,  # desactivado — dados mensais não têm padrão semanal
                    daily_seasonality=False,   # desactivado — dados mensais não têm padrão diário
                    **params
                )
                for col in prophet_features:
                    m.add_regressor(col)  # adicionar macro features como regressores externos
                m.fit(df_tr)
                fc = m.predict(df_vl[['ds'] + prophet_features])
                preds.extend(fc['yhat'].values)
                actuals.extend(df_vl['y'].values)
            except:
                continue

        if preds:
            fold_rmses.append(np.sqrt(mean_squared_error(actuals, preds)))

    result = params.copy()
    result['mean_rmse'] = np.mean(fold_rmses) if fold_rmses else np.nan
    result['std_rmse']  = np.std(fold_rmses)  if fold_rmses else np.nan
    prophet_random_results.append(result)

    print(f"Trial {i+1}/{n_trials} | Mean RMSE: {result['mean_rmse']:,.2f} | {params}")
    print("-" * 60)

prophet_random_seg = pd.DataFrame(prophet_random_results).sort_values('mean_rmse').reset_index(drop=True)
prophet_random_seg.head(10)

##### ETS

##### 7.2.2.4 Comparison

In [ ]:
print("=== Melhores Parâmetros ===")

print("\nXGBoost:")
print(xgb_random_seg.iloc[0][['n_estimators','max_depth','learning_rate',
                               'subsample','colsample_bytree','min_child_weight']].to_dict())

print("\nLightGBM:")
print(lgbm_random_seg.iloc[0][['n_estimators','num_leaves','learning_rate',
                                'subsample','colsample_bytree','min_child_samples',
                                'reg_alpha','reg_lambda']].to_dict())

print("\nProphet:")
print(prophet_random_seg.iloc[0][['changepoint_prior_scale','seasonality_prior_scale',
                                  'seasonality_mode','changepoint_range']].to_dict())

comparison = pd.DataFrame({
    'Model': ['XGBoost', 'LightGBM', 'Prophet'],
    'Mean RMSE': [
        xgb_random_seg.iloc[0]['mean_rmse'],
        lgbm_random_seg.iloc[0]['mean_rmse'],
        prophet_random_seg.iloc[0]['mean_rmse']
    ]
}).sort_values('Mean RMSE').reset_index(drop=True)

print("\n=== RMSE por Modelo ===")
print(comparison.to_string(index=False))

#### 7.2.3 Business Unit Level

##### 7.2.3.1 XGBoost

In [ ]:
n_trials = 20  # número de combinações aleatórias a testar — aumentar para explorar mais
xgb_random_results = []

for i in range(n_trials):
    params = {
        'n_estimators'    : random.choice([150, 200, 250, 300, 350]),  # nº de árvores — mais = mais lento mas potencialmente melhor
        'max_depth'       : random.choice([3, 4, 5, 6]),               # profundidade de cada árvore — mais alto = mais overfitting
        'learning_rate'   : random.choice([0.03, 0.05, 0.07, 0.1]),   # tamanho do passo — mais baixo precisa de mais árvores
        'subsample'       : random.choice([0.7, 0.8, 0.9]),            # % de linhas usadas por árvore — ajuda a regularizar
        'colsample_bytree': random.choice([0.7, 0.8, 0.9]),            # % de features usadas por árvore — ajuda a regularizar
        'min_child_weight': random.choice([1, 3, 5, 7]),               # mínimo de peso por folha — mais alto = menos overfitting
    }

    fold_rmses = []
    for fold in expanding_folds:
        train_df, val_df = get_expanding_fold(
            df_bu_final, fold['train_end'], fold['val_start'], fold['val_end']
        )
        model = XGBRegressor(objective='reg:squarederror', random_state=42, n_jobs=-1, **params)
        model.fit(train_df[final_features], train_df['y'])
        y_pred = model.predict(val_df[final_features])
        fold_rmses.append(np.sqrt(mean_squared_error(val_df['y'], y_pred)))

    result = params.copy()
    result['mean_rmse'] = np.mean(fold_rmses)
    result['std_rmse']  = np.std(fold_rmses)
    xgb_random_results.append(result)

    print(f"Trial {i+1}/{n_trials} | Mean RMSE: {result['mean_rmse']:,.2f} | {params}")
    print("-" * 60)

xgb_random_bu = pd.DataFrame(xgb_random_results).sort_values('mean_rmse').reset_index(drop=True)
xgb_random_bu.head(10)

##### 7.2.3.2 LightGBM

In [ ]:
n_trials = 20  # número de combinações aleatórias a testar — aumentar para explorar mais
lgbm_random_results = []

for i in range(n_trials):
    params = {
        'n_estimators'     : random.choice([150, 200, 250, 300, 350]),  # nº de árvores — mais = mais lento mas potencialmente melhor
        'num_leaves'       : random.choice([15, 31, 50, 63]),           # complexidade de cada árvore — valores altos = mais overfitting
        'learning_rate'    : random.choice([0.03, 0.05, 0.07, 0.1]),   # tamanho do passo — mais baixo precisa de mais árvores
        'subsample'        : random.choice([0.7, 0.8, 0.9]),            # % de linhas usadas por árvore — ajuda a regularizar
        'colsample_bytree' : random.choice([0.7, 0.8, 0.9]),            # % de features usadas por árvore — ajuda a regularizar
        'min_child_samples': random.choice([5, 10, 20, 30]),            # mínimo de obs por folha — mais alto = menos overfitting
        'reg_alpha'        : random.choice([0, 0.01, 0.1]),             # regularização L1 — penaliza coeficientes pequenos
        'reg_lambda'       : random.choice([0, 0.01, 0.1]),             # regularização L2 — penaliza coeficientes grandes
    }

    fold_rmses = []
    for fold in expanding_folds:
        train_df, val_df = get_expanding_fold(
            df_bu_final, fold['train_end'], fold['val_start'], fold['val_end']
        )
        model = lgb.LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1, **params)
        model.fit(train_df[final_features], train_df['y'])
        y_pred = model.predict(val_df[final_features])
        fold_rmses.append(np.sqrt(mean_squared_error(val_df['y'], y_pred)))

    result = params.copy()
    result['mean_rmse'] = np.mean(fold_rmses)
    result['std_rmse']  = np.std(fold_rmses)
    lgbm_random_results.append(result)

    print(f"Trial {i+1}/{n_trials} | Mean RMSE: {result['mean_rmse']:,.2f} | {params}")
    print("-" * 60)

lgbm_random_bu = pd.DataFrame(lgbm_random_results).sort_values('mean_rmse').reset_index(drop=True)
lgbm_random_bu.head(10)


##### 7.2.3.3 Prophet

In [ ]:
logging.getLogger('prophet').setLevel(logging.WARNING)  # silenciar logs do Prophet

n_trials = 20  # número de combinações aleatórias a testar
prophet_random_results = []
scale = 1e6  # escalar para evitar overflow numérico no Stan

for i in range(n_trials):
    params = {
        'changepoint_prior_scale' : random.choice([0.001, 0.01, 0.05, 0.1, 0.3]),
        'seasonality_prior_scale' : random.choice([0.01, 0.1, 1.0, 5.0, 10.0]),
        'seasonality_mode'        : random.choice(['additive', 'multiplicative']),
        'changepoint_range'       : random.choice([0.8, 0.85, 0.9]),
    }

    fold_rmses = []
    for fold in expanding_folds:
        train_df, val_df = get_expanding_fold(
            df_bu_final, fold['train_end'], fold['val_start'], fold['val_end']
        )

        preds, actuals = [], []
        for uid in train_df['unique_id'].unique():
            ts = train_df[train_df['unique_id'] == uid].copy()
            vs = val_df[val_df['unique_id'] == uid].copy()
            if len(vs) == 0:
                continue

            df_tr = ts[['Period','y']].rename(columns={'Period':'ds'})
            df_vl = vs[['Period','y']].rename(columns={'Period':'ds'})
            df_tr['ds'] = df_tr['ds'].apply(lambda p: pd.Timestamp('2019-01-01') + pd.DateOffset(months=int(p)-1))
            df_vl['ds'] = df_vl['ds'].apply(lambda p: pd.Timestamp('2019-01-01') + pd.DateOffset(months=int(p)-1))
            df_tr['y'] = df_tr['y'].astype(float) / scale
            df_vl['y'] = df_vl['y'].astype(float) / scale
            for col in prophet_features:
                df_tr[col] = ts[col].values.astype(float)
                df_vl[col] = vs[col].values.astype(float)

            try:
                m = Prophet(
                    yearly_seasonality=True,
                    weekly_seasonality=False,
                    daily_seasonality=False,
                    **params
                )
                for col in prophet_features:
                    m.add_regressor(col)
                m.fit(df_tr)
                fc = m.predict(df_vl[['ds'] + prophet_features])
                preds.extend(fc['yhat'].values * scale)    # reescalar para euros
                actuals.extend(df_vl['y'].values * scale)  # reescalar para euros
            except Exception as e:
                print(f"  ERRO série {uid}: {e}")
                continue

        if preds:
            fold_rmses.append(np.sqrt(mean_squared_error(actuals, preds)))

    result = params.copy()
    result['mean_rmse'] = np.mean(fold_rmses) if fold_rmses else np.nan
    result['std_rmse']  = np.std(fold_rmses)  if fold_rmses else np.nan
    prophet_random_results.append(result)

    print(f"Trial {i+1}/{n_trials} | Mean RMSE: {result['mean_rmse']:,.2f} | {params}")
    print("-" * 60)

prophet_random_bu = pd.DataFrame(prophet_random_results).sort_values('mean_rmse').reset_index(drop=True)
prophet_random_bu.head(10)

##### ETS

In [ ]:
""" # Parâmetros possíveis do ETS — cada modelo é definido por 3 letras (Erro + Tendência + Sazonalidade):
#   Erro:       A = aditivo,        M = multiplicativo
#   Tendência:  A = aditiva,        N = sem tendência
#   Sazonalidade: A = aditiva,      M = multiplicativa,   N = sem sazonalidade
# Exemplo: 'AAN' = erro aditivo, tendência aditiva, sem sazonalidade
error_types  = ['A', 'M']
trend_types  = ['A', 'N']
season_types = ['A', 'M', 'N']

n_trials = 20
ets_random_results = []

for i in range(n_trials):
    error  = random.choice(error_types)
    trend  = random.choice(trend_types)
    season = random.choice(season_types)
    # damped amorte a tendência ao longo do tempo — só faz sentido se houver tendência
    damped = random.choice([True, False]) if trend != 'N' else False

    params = {'model': error + trend + season, 'damped': damped}

    fold_rmses = []
    for fold in expanding_folds:
        train_df, val_df = get_expanding_fold(
            df_bu_final, fold['train_end'], fold['val_start'], fold['val_end']
        )

        # ETS usa apenas unique_id, ds, y — sem features externas
        Y_tr  = train_df[['unique_id', 'Period', 'y']].rename(columns={'Period': 'ds'})
        Y_val = val_df[['unique_id', 'Period', 'y']].rename(columns={'Period': 'ds'})

        try:
            sf = StatsForecast(
                models=[ETS(season_length=12, **params)],
                freq=1, n_jobs=-1
            )
            sf.fit(Y_tr)
            preds  = sf.predict(h=int(Y_val['ds'].nunique())).reset_index()
            col    = [c for c in preds.columns if c not in ['unique_id', 'ds']][0]
            merged = Y_val.merge(preds, on=['unique_id', 'ds'], how='left').dropna(subset=[col])
            fold_rmses.append(np.sqrt(mean_squared_error(merged['y'], merged[col])))
        except Exception:
            continue  # alguns modelos falham com séries muito curtas ou com valores negativos

    result = {**params, 'mean_rmse': np.mean(fold_rmses) if fold_rmses else np.nan}
    ets_random_results.append(result)
    print(f"Trial {i+1}/{n_trials} | RMSE: {result['mean_rmse']:,.2f} | {params}")

ets_random_bu = pd.DataFrame(ets_random_results).sort_values('mean_rmse').reset_index(drop=True)
ets_random_bu.head(10)"""

##### 7.2.3.4 Comparison

In [ ]:
print("=== Melhores Parâmetros ===")

print("\nXGBoost:")
print(xgb_random_bu.iloc[0][['n_estimators','max_depth','learning_rate',
                               'subsample','colsample_bytree','min_child_weight']].to_dict())

print("\nLightGBM:")
print(lgbm_random_bu.iloc[0][['n_estimators','num_leaves','learning_rate',
                                'subsample','colsample_bytree','min_child_samples',
                                'reg_alpha','reg_lambda']].to_dict())

print("\nProphet:")
print(prophet_random_bu.iloc[0][['changepoint_prior_scale','seasonality_prior_scale',
                                  'seasonality_mode','changepoint_range']].to_dict())

comparison = pd.DataFrame({
    'Model': ['XGBoost', 'LightGBM', 'Prophet'],
    'Mean RMSE': [
        xgb_random_bu.iloc[0]['mean_rmse'],
        lgbm_random_bu.iloc[0]['mean_rmse'],
        prophet_random_bu.iloc[0]['mean_rmse']
    ]
}).sort_values('Mean RMSE').reset_index(drop=True)

print("\n=== RMSE por Modelo ===")
print(comparison.to_string(index=False))

### 7.3 Final Models

In [ ]:
def build_future_row(history_y, last_known_exog, next_period):
    """
    history_y: lista com histórico real + previsões já geradas
    last_known_exog: dict com valores das features exógenas a congelar
    next_period: período futuro a prever
    """
    row = {}

    # lags da y
    row['y_lag_1'] = history_y[-1]
    row['y_lag_3'] = history_y[-3]
    row['y_lag_6'] = history_y[-6]

    # rolling means
    row['y_roll_mean_3'] = np.mean(history_y[-3:])
    row['y_roll_mean_6'] = np.mean(history_y[-6:])
    row['y_roll_mean_9'] = np.mean(history_y[-9:])

    # exógenas "congeladas" no último valor observado
    for k, v in last_known_exog.items():
        row[k] = v

    row['Period'] = next_period
    return row

In [ ]:
def recursive_ml_forecast(
    df_level,
    feature_cols,
    horizon,
    model_type='xgb',
    model_params=None
):
    """
    df_level: dataframe de um nível (subseg, seg, bu)
    com colunas unique_id, Period, y e feature_cols.
    """
    forecasts = []

    if model_params is None:
        model_params = {}

    for uid in df_level['unique_id'].unique():
        df_series = df_level[df_level['unique_id'] == uid].sort_values('Period').copy()

        X_train = df_series[feature_cols]
        y_train = df_series['y']

        if model_type == 'xgb':
            model = XGBRegressor(
                objective='reg:squarederror',
                random_state=42,
                n_jobs=-1,
                **model_params
            )
        elif model_type == 'lgbm':
            model = lgb.LGBMRegressor(
                random_state=42,
                **model_params
            )
        else:
            raise ValueError("model_type must be 'xgb' or 'lgbm'")

        model.fit(X_train, y_train)

        # histórico da y para forecast recursivo
        history_y = list(y_train.values)

        # congelar apenas as exógenas que estão nas final_features e não são y_lag/roll
        exog_cols = [
            c for c in feature_cols
            if not c.startswith('y_lag_') and not c.startswith('y_roll_')
        ]
        last_known_exog = df_series[exog_cols].iloc[-1].to_dict()

        last_period = df_series['Period'].max()

        preds = []
        future_periods = []

        for h in range(1, horizon + 1):
            next_period = last_period + h
            row = build_future_row(history_y, last_known_exog, next_period)

            X_future = pd.DataFrame([row])[feature_cols]
            y_pred = model.predict(X_future)[0]

            preds.append(y_pred)
            future_periods.append(next_period)

            history_y.append(y_pred)

        df_preds = pd.DataFrame({
            'unique_id': uid,
            'ds': future_periods,
            'yhat': preds
        })

        forecasts.append(df_preds)

    forecasts = pd.concat(forecasts, ignore_index=True)
    return forecasts

In [ ]:
TRAIN_END = 36
TEST_START = 37
TEST_END = 42
H_EVAL = TEST_END - TRAIN_END  # 6 períodos
 
print(f"\nPreparar dados de treino (períodos 1-{TRAIN_END})...")
 
# Filtrar dados de treino para cada nível
train_subseg_eval = df_subseg_final[df_subseg_final['Period'] <= TRAIN_END].copy()
train_seg_eval = df_seg_final[df_seg_final['Period'] <= TRAIN_END].copy()
train_bu_eval = df_bu_final[df_bu_final['Period'] <= TRAIN_END].copy()
 
print(f"  Subsegments: {train_subseg_eval['unique_id'].nunique()} séries, {len(train_subseg_eval)} obs")
print(f"  Segments: {train_seg_eval['unique_id'].nunique()} séries, {len(train_seg_eval)} obs")
print(f"  BUs: {train_bu_eval['unique_id'].nunique()} séries, {len(train_bu_eval)} obs")

In [ ]:
best_xgb_params_sub = {
    'n_estimators': 250,
    'max_depth': 3, 
    'learning_rate': 0.03, 
    'subsample': 0.7, 
    'colsample_bytree': 0.8, 
    'min_child_weight': 3
}
 
best_lgbm_params_seg = {
    'n_estimators': 150,
    'num_leaves': 50, 
    'learning_rate': 0.03, 
    'subsample': 0.8, 
    'colsample_bytree': 0.8, 
    'min_child_samples': 10, 
    'reg_alpha': 0.1, 
    'reg_lambda': 0
}
 
best_lgbm_params_bu = {
    'n_estimators': 200,
    'num_leaves': 15,
    'learning_rate': 0.1,
    'subsample': 0.8, 
    'colsample_bytree': 0.8, 
    'min_child_samples': 10, 
    'reg_alpha': 0.01, 
    'reg_lambda': 0.1
}
 

#### 7.3.1 Subsegment Level

In [ ]:
eval_subseg_fcst = recursive_ml_forecast(
    df_level=train_subseg_eval,  # Dados até período 36
    feature_cols=final_features,
    horizon=H_EVAL,
    model_type='xgb',
    model_params=best_xgb_params_sub
)
print(f"    ✓ {eval_subseg_fcst['unique_id'].nunique()} séries × {H_EVAL} períodos")

#### 7.3.2 Segment Level

In [ ]:
print("  Segments (LightGBM)...")
eval_seg_fcst = recursive_ml_forecast(
    df_level=train_seg_eval,  # Dados até período 36
    feature_cols=final_features,
    horizon=H_EVAL,
    model_type='lgbm',
    model_params=best_lgbm_params_seg
)
print(f"    ✓ {eval_seg_fcst['unique_id'].nunique()} séries × {H_EVAL} períodos")

#### 7.3.3 Business Unit Level

In [ ]:
print("  Business Units (LightGBM)...")
eval_bu_fcst = recursive_ml_forecast(
    df_level=train_bu_eval,  # Dados até período 36
    feature_cols=final_features,
    horizon=H_EVAL,
    model_type='lgbm',
    model_params=best_lgbm_params_bu
)
print(f"    ✓ {eval_bu_fcst['unique_id'].nunique()} séries × {H_EVAL} períodos")

## 8. Reconciliation

### 8.1 Join Final Predictions

In [ ]:
eval_base_fcst = pd.concat([eval_bu_fcst, eval_seg_fcst, eval_subseg_fcst], ignore_index=True)

In [ ]:

# Adicionar TOTAL como soma dos subsegments
eval_total_fcst = eval_subseg_fcst.groupby('ds')['yhat'].sum().reset_index()
eval_total_fcst['unique_id'] = 'TOTAL'
eval_base_fcst = pd.concat([eval_total_fcst[['unique_id', 'ds', 'yhat']], eval_base_fcst], ignore_index=True)
 
print(f"\nTotal: {eval_base_fcst['unique_id'].nunique()} séries")

### 8.2 Apply MinTrace

In [ ]:
print("\n--- Aplicar MinTrace OLS ---")
 
# Pivot: linhas = unique_id, colunas = período
Y_hat_eval = eval_base_fcst.pivot(index='unique_id', columns='ds', values='yhat')
Y_hat_eval = Y_hat_eval.reindex(all_nodes_list)
 
print(f"  Matriz Y_hat: {Y_hat_eval.shape}")
 
# Aplicar MinTrace (S_df e reconcile_mintrace_ols vêm da Secção 5)
Y_recon_eval = reconcile_mintrace_ols(Y_hat_eval.values, S_df.values)
Y_recon_eval = pd.DataFrame(Y_recon_eval, index=all_nodes_list, columns=Y_hat_eval.columns)
 
print(f"Reconciliado: {Y_recon_eval.shape}")

### Testing our MinTrace

In [ ]:
# Filtrar dados de teste
test_subseg = df_subseg_final[(df_subseg_final['Period'] >= TEST_START) & 
                               (df_subseg_final['Period'] <= TEST_END)].copy()
test_seg = df_seg_final[(df_seg_final['Period'] >= TEST_START) & 
                         (df_seg_final['Period'] <= TEST_END)].copy()
test_bu = df_bu_final[(df_bu_final['Period'] >= TEST_START) & 
                       (df_bu_final['Period'] <= TEST_END)].copy()

In [ ]:
real_subseg = test_subseg[['unique_id', 'Period', 'y']].rename(columns={'Period': 'ds', 'y': 'actual'})
real_seg = test_seg[['unique_id', 'Period', 'y']].rename(columns={'Period': 'ds', 'y': 'actual'})
real_bu = test_bu[['unique_id', 'Period', 'y']].rename(columns={'Period': 'ds', 'y': 'actual'})
 
real_all = pd.concat([real_bu, real_seg, real_subseg], ignore_index=True)
 
# Adicionar TOTAL
real_total = real_subseg.groupby('ds')['actual'].sum().reset_index()
real_total['unique_id'] = 'TOTAL'
real_all = pd.concat([real_total[['unique_id', 'ds', 'actual']], real_all], ignore_index=True)
 
# Pivot
Y_actual_eval = real_all.pivot(index='unique_id', columns='ds', values='actual')
Y_actual_eval = Y_actual_eval.reindex(all_nodes_list)
 
print(f"  Valores reais: {Y_actual_eval.shape}")

In [ ]:
def calc_rmse(actual, predicted):
    """Calcular RMSE ignorando NaNs"""
    actual_flat = actual.values.flatten().astype(float)
    pred_flat = predicted.values.flatten().astype(float)
    mask = ~(np.isnan(actual_flat) | np.isnan(pred_flat))
    if mask.sum() == 0:
        return np.nan
    return np.sqrt(np.mean((actual_flat[mask] - pred_flat[mask])**2))
 
# Níveis da hierarquia
levels = {
    'Total': ['TOTAL'],
    'Business Unit': [n for n in all_nodes_list if n.startswith('BU__')],
    'Segment': [n for n in all_nodes_list if n.startswith('SEG__')],
    'Subsegment': [n for n in all_nodes_list if n.startswith('SUBSEG__')],
}

In [ ]:
eval_results = []
for level_name, nodes in levels.items():
    rmse_base = calc_rmse(Y_actual_eval.loc[nodes], Y_hat_eval.loc[nodes])
    rmse_recon = calc_rmse(Y_actual_eval.loc[nodes], Y_recon_eval.loc[nodes])
    improvement = ((rmse_base - rmse_recon) / rmse_base) * 100 if rmse_base > 0 else 0
    
    eval_results.append({
        'Level': level_name,
        'Base_RMSE': rmse_base,
        'MinTrace_RMSE': rmse_recon,
        'Improvement_%': improvement
    })

In [ ]:
# RMSE global
rmse_base_all = calc_rmse(Y_actual_eval, Y_hat_eval)
rmse_recon_all = calc_rmse(Y_actual_eval, Y_recon_eval)
improvement_all = ((rmse_base_all - rmse_recon_all) / rmse_base_all) * 100
 
print("-" * 60)
print(f"{'GLOBAL':<15} {rmse_base_all:>15,.0f} {rmse_recon_all:>15,.0f} {improvement_all:>11.1f}%")
 
# Guardar resultados
eval_results_df = pd.DataFrame(eval_results)
eval_results_df.to_csv('evaluation_results_mintrace.csv', index=False)
print("\n✓ Guardado: evaluation_results_mintrace.csv")